# GRB & Kilonova Mass-Plane Diagnostics

Extending the Gottlieb et al. ([2023](https://arxiv.org/abs/2309.00038)) GRB classification framework with the kilonova connection from Gottlieb et al. ([2024](https://arxiv.org/abs/2411.13657v2)).

**2024 update (arXiv:2411.13657v2):** The sub-threshold ($M_\mathrm{tot} < M_\mathrm{thresh}$) region is split by HMNS lifetime:
- **Long-lived HMNS** ($M_\mathrm{tot} \lesssim 1.2\,M_\mathrm{TOV}$, $t_\mathrm{HMNS} \sim 0.1\text{ to }1\,$s): powers **sbGRB + blue KN**
- **Short-lived HMNS** ($1.2\,M_\mathrm{TOV} \lesssim M_\mathrm{tot} < M_\mathrm{thresh}$, $t_\mathrm{HMNS} \sim 10\text{ to }100\,$ms): collapses to BH with massive disk, powers **lbGRB + red KN**

Kilonova color (red vs blue) becomes the diagnostic for the sbGRB central engine.

**Data:** COMPAS fiducial BNS/BHNS populations (Model A, Broekgaarden et al. 2021/2022).

**Cosmology:** All redshift–time–volume conversions use the COMPAS `FastCosmicIntegration` default parameters (Planck 2015 / TNG-consistent): $H_0 = 67.74$ km/s/Mpc, $\Omega_m = 0.3089$, $\Omega_\Lambda = 0.6911$. Mixing with Planck 2018 values introduces $\sim$2% inconsistencies in lookback times at high $z$.

**SN remnant prescription:** The COMPAS runs (Models A and K) use the Fryer et al. (2012) *rapid* supernova explosion mechanism. This produces a narrow NS mass distribution peaked near $1.26$–$1.28\,M_\odot$ with a mass gap around $2$–$5\,M_\odot$. The *delayed* mechanism yields broader NS masses and would shift class fractions (especially near the $1.2 \times M_\mathrm{TOV}$ boundary). This is a systematic uncertainty.

**Jet efficiency caveat:** All disk-mass-based classifications assume 100% jet launching efficiency above the disk-mass threshold. In practice, the jet must break out of the ejecta and the disk must reach the MAD state (Gottlieb 2023). Predicted GRB rates should be interpreted as **upper bounds**.

**Note on sbGRB (blue KN) systems:** With $M_\mathrm{TOV} = 2.20\,M_\odot$ (Raaijmakers et al. 2021, arXiv:2105.06981, PP model: $M_\mathrm{TOV} = 2.23^{+0.14}_{-0.23}$), the HMNS split sits at $1.2\,M_\mathrm{TOV} \approx 2.64\,M_\odot$. The COMPAS Model A NS mass distribution, which uses the Fryer et al. (2012) rapid supernova prescription, is peaked near $1.26$–$1.28\,M_\odot$, so equal-mass BNS pairs with $M_\mathrm{tot} \approx 2.52\,M_\odot$ fall *below* the HMNS split and are predicted to power **sbGRB + blue KN**. The actual sbGRB fraction is small because the rapid mechanism's narrow NS-mass distribution combined with the natal-kick survival bias preferentially populates the heavier end ($M_\mathrm{tot} \gtrsim 2.6\,M_\odot$). The cell below reports the empirical fraction in this run; it is sensitive to the SN remnant prescription and to $M_\mathrm{TOV}$ at the few-percent level. Observed Galactic double neutron star systems (e.g., J1756$-$2251, B1534+12) demonstrate that lighter BNS pairs do exist (see also Gottlieb 2024, Figure 3).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from matplotlib.lines import Line2D
from matplotlib.colors import ListedColormap, BoundaryNorm, to_rgba
from matplotlib.patches import Wedge, Patch
from scipy.interpolate import interp1d

from grb_physics import (foucart_disk_mass, M_CRIT_BNS, Q_THRESH_BNS,
                          MDISK_SHORT, MDISK_LONG, M_TOV, M_THRESH,
                          EOS_MODELS, mcrit_to_r14,
                          bns_dynamical_ejecta, bhns_dynamical_ejecta,
                          MISALIGNMENT_SYSTEMATIC_FACTOR,
                          effective_aligned_spin, ns_radius_from_eos)
from grb_classify import (classify_bns_2023, classify_bns_2024, classify_bhns,
                           classify_grid, classify_formation_channels)
from grb_rates import (compute_merger_rate, per_system_rate_weights,
                        calibrate_mean_mass_evolved,
                        formation_efficiency, marginalize,
                        rate_label,
                        beamed_rate, OBSERVED_SGRB_RATES,
                        wanderman_piran_2015_Rz,
                        check_dPdlogZ_normalization, CLASS_THETA_J,
                        apply_bhns_misalignment)
from grb_io import (load_bns, load_bhns, load_bns_with_channels,
                     load_bns_with_kicks, load_bhns_with_kicks,
                     read_expected_local_rate, read_metallicity_range,
                     verify_shared_metallicity_prior,
                     METALLICITY_GRID, weighted_sample, log_jitter,
                     DEFAULT_BNS_PATH, DEFAULT_BHNS_PATH, DEFAULT_BNS_K_PATH)

plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 150,
                     'savefig.bbox': 'tight', 'font.size': 10})

A_BH_FID = 0.5
M_HMNS_SPLIT = 1.2 * M_TOV

# COMPAS Model A maximum NS gravitational mass (Broekgaarden+ 2021,
# arXiv:2103.02608; their fiducial M_NS_max = 2.5 Msun, Models J/K = 2.0/3.0).
# This is the upper edge of the BNS region on the (M1, M2) plane and is
# threaded through every classify_grid call below.
NS_MAX_BNS = 2.5

C_SB_BLUE   = '#06B6D4'
C_LB_HMNS   = '#DC2626'
C_LB_DISK   = '#DC2626'
C_FAINT     = '#F59E0B'
C_NO_GRB    = '#334155'
C_FAINT_BH  = '#F59E0B'
C_LONG_BH   = '#DC2626'

# ── Load BNS (with channels + kicks for later sections) ───────────────────
bns_ch = load_bns_with_channels()
bns_k  = load_bns_with_kicks()

m1_bns, m2_bns = bns_ch['m1'], bns_ch['m2']
w_bns  = bns_ch['weights']
Z_bns  = bns_ch['metallicity']
delay_bns = bns_ch['delay_time']
n_bns  = bns_ch['n_merging']

M_tot  = m1_bns + m2_bns
q_bns  = m1_bns / m2_bns

# ── BNS classification ────────────────────────────────────────────────────
cls24 = classify_bns_2024(m1_bns, m2_bns)
sbGRB_blue  = cls24['sbGRB + blue KN']
lbGRB_hmns  = cls24['lbGRB + red KN (HMNS)']
lbGRB_disk  = cls24['lbGRB + red KN (disk)']
bns_faint_lb  = cls24['Faint lbGRB']
faint_or_none = bns_faint_lb

cls23 = classify_bns_2023(m1_bns, m2_bns)
short_typeI_23  = cls23['Short Type-I']
short_typeII_23 = cls23['Short Type-II']
long_cbGRB_23   = cls23['Long cbGRB']

# ── Load BHNS (with kicks) ────────────────────────────────────────────────
bhns = load_bhns_with_kicks()
BH, NS_bh = bhns['M_BH'], bhns['M_NS']
w_bhns     = bhns['weights']
Z_bhns     = bhns['metallicity']
delay_bhns = bhns['delay_time']
n_bhns     = bhns['n_merging']

# ── BHNS classification ───────────────────────────────────────────────────
cbhns = classify_bhns(BH, NS_bh, a_BH=A_BH_FID)
M_disk_bhns   = cbhns['M_disk']
bhns_no_grb   = cbhns['No GRB']
bhns_faint_lb = cbhns['Short cbGRB']
bhns_long     = cbhns['Long cbGRB']

# ── Kick data ─────────────────────────────────────────────────────────────
v_sys_bns  = bns_k['v_sys']
v_sys_bhns = bhns['v_sys']

print(f"BNS  merging systems: {n_bns:,}")
for lbl, m in [('sbGRB+blue KN (long-lived HMNS)', sbGRB_blue),
               ('lbGRB+red KN  (short-lived HMNS)', lbGRB_hmns),
               ('lbGRB+red KN  (massive disk)', lbGRB_disk),
               ('Faint lbGRB   (small disk / prompt collapse)', bns_faint_lb)]:
    print(f"  {lbl:44s}: {m.sum():>7,}  ({100*m.sum()/n_bns:.1f}%)")
print(f"\nBHNS merging systems: {n_bhns:,}  (a_BH = {A_BH_FID})")
for lbl, m in [('lbGRB+red KN  (massive disk)', bhns_long),
               ('Faint lbGRB   (small disk)', bhns_faint_lb),
               ('No GRB/KN     (NS swallowed)', bhns_no_grb)]:
    print(f"  {lbl:44s}: {m.sum():>7,}  ({100*m.sum()/n_bhns:.1f}%)")


## 1. Mass Plane, BNS + BHNS (cf. Gottlieb et al. 2024, Figure 3)

Combined $M_1$ vs $M_2$ diagram spanning all compact binary merger types. The background colormap shows the predicted GRB + KN outcome; COMPAS populations are overlaid as scatter points. The lower-left triangle is the NS to NS domain; the upper region is the BHNS domain (classified via the Foucart disk mass at $a_\mathrm{BH} = 0.5$).

In [ ]:
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LogNorm, LinearSegmentedColormap

cmap_hex = LinearSegmentedColormap.from_list('purple', ['#000000', '#A855F7'])

# ── build grid ─────────────────────────────────────────────────────────────
# Use NS_MAX_BNS (Broekgaarden+ 2021 Model A M_NS_max = 2.5 Msun) as the
# BNS/BHNS boundary on the grid, matching the COMPAS run configuration.
ns_max = NS_MAX_BNS
m2_lo, m2_hi = 1.25, 2.55
m1_lo_bns, m1_hi_bns = 1.25, ns_max
m1_lo_bhns, m1_hi_bhns = ns_max, 20
n2 = 700

m2_arr_bns = np.linspace(m2_lo, m2_hi, n2)
m1_arr_bns = np.linspace(m1_lo_bns, m1_hi_bns, 600)
M2G_bns, M1G_bns = np.meshgrid(m2_arr_bns, m1_arr_bns)
cls_bns = classify_grid(M1G_bns, M2G_bns, a_bh=A_BH_FID, ns_max=NS_MAX_BNS)
cls_bns_masked = np.ma.array(cls_bns, mask=(M1G_bns < M2G_bns))

m2_arr_bhns = np.linspace(m2_lo, m2_hi, n2)
m1_arr_bhns = np.linspace(m1_lo_bhns, m1_hi_bhns, 600)
M2G_bhns, M1G_bhns = np.meshgrid(m2_arr_bhns, m1_arr_bhns)
cls_bhns = classify_grid(M1G_bhns, M2G_bhns, a_bh=A_BH_FID, ns_max=NS_MAX_BNS)
cls_bhns_masked = np.ma.array(cls_bhns, mask=(M2G_bhns < 0.8))

# ── colormaps ──────────────────────────────────────────────────────────────
grid_colors = [C_NO_GRB, C_FAINT, C_LB_HMNS, C_SB_BLUE, C_LB_DISK, C_FAINT_BH, C_LONG_BH]
cmap_grid = ListedColormap([to_rgba(c, 0.60) for c in grid_colors])
norm_grid = BoundaryNorm(np.arange(-0.5, 7.5, 1), cmap_grid.N)

# ── figure (full width for plots, legends below) ──────────────────────────
fig = plt.figure(figsize=(10, 13))
gs = GridSpec(2, 1, height_ratios=[1, 1], hspace=0.08, figure=fig)
ax_bh = fig.add_subplot(gs[0])
ax_ns = fig.add_subplot(gs[1], sharex=ax_bh)

# ═══════════════════════ Upper panel: BH-NS ════════════════════════════════
ax_bh.pcolormesh(m2_arr_bhns, m1_arr_bhns, cls_bhns_masked,
                 cmap=cmap_grid, norm=norm_grid, shading='nearest',
                 rasterized=True, zorder=0)

_MBH_c = np.linspace(m1_lo_bhns, m1_hi_bhns, 300)
_MNS_c = np.linspace(0.9, m2_hi, 250)
_MNSg, _MBHg = np.meshgrid(_MNS_c, _MBH_c)
_disk_g = foucart_disk_mass(_MBHg, _MNSg, a_BH=A_BH_FID)

ax_bh.contour(_MNSg, _MBHg, (_disk_g > 0).astype(float), levels=[0.5],
              colors='#1E293B', linewidths=1.8, linestyles='--', zorder=7)
_dg_safe = np.where(_disk_g > 0, _disk_g, np.nan)
ax_bh.contour(_MNSg, _MBHg, _dg_safe,
              levels=[MDISK_SHORT, MDISK_LONG],
              colors=['#92400E', '#7F1D1D'], linewidths=1.8,
              linestyles=':', zorder=7)

hb_bh = ax_bh.hexbin(NS_bh, BH, gridsize=120, cmap=cmap_hex,
                      mincnt=1, norm=LogNorm(), zorder=2,
                      edgecolors='none', linewidths=0.2, alpha=0.85)

ax_bh.set_ylabel(r'$M_\mathrm{BH}\ [M_\odot]$', fontsize=12)
ax_bh.set_ylim(m1_lo_bhns, m1_hi_bhns)
ax_bh.tick_params(labelbottom=False, labelsize=10)
ax_bh.set_axisbelow(True)
ax_bh.text(0.95, 0.95, 'BH\u2013NS', fontsize=10, fontweight='bold',
           color='#1E293B', transform=ax_bh.transAxes, va='top', ha='right',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='none', alpha=0.95))
ax_bh.text(0.87, 0.08, rf'$a_{{\mathrm{{BH}}}} = {A_BH_FID}$',
           fontsize=9, fontweight='bold', color='#1E293B', transform=ax_bh.transAxes,
           va='top',
           bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=1.0))

# ═══════════════════════ Lower panel: NS-NS ════════════════════════════════
ax_ns.pcolormesh(m2_arr_bns, m1_arr_bns, cls_bns_masked,
                 cmap=cmap_grid, norm=norm_grid, shading='nearest',
                 rasterized=True, zorder=0)

hb_ns = ax_ns.hexbin(m2_bns, m1_bns, gridsize=120, cmap=cmap_hex,
                      mincnt=1, norm=LogNorm(), zorder=2,
                      edgecolors='none', linewidths=0.2, alpha=0.85)

_m2 = np.linspace(m2_lo, m2_hi, 500)

_mt = M_THRESH - _m2
_ok = (_mt >= m1_lo_bns) & (_mt <= m1_hi_bns) & (_m2 <= _mt)
ax_ns.plot(_m2[_ok], _mt[_ok], color='#0F172A', lw=2.0, ls='--', zorder=8)

_mh = M_HMNS_SPLIT - _m2
_ok2 = (_mh >= m1_lo_bns) & (_mh <= m1_hi_bns) & (_m2 <= _mh)
ax_ns.plot(_m2[_ok2], _mh[_ok2], color='#0E7490', lw=1.8, ls='-.', zorder=8)

_ql = _m2 * Q_THRESH_BNS
_ok3 = (_ql >= m1_lo_bns) & (_ql <= m1_hi_bns) & (_m2 <= _ql)
ax_ns.plot(_m2[_ok3], _ql[_ok3], color='#0F172A', lw=1.6, ls=':', zorder=8)

diag_lo, diag_hi = m2_lo, min(m2_hi, m1_hi_bns)
ax_ns.plot([diag_lo, diag_hi], [diag_lo, diag_hi],
           color='#94A3B8', lw=0.7, ls=(0, (4, 3)), alpha=0.6, zorder=1)

for (mx, my, name, ha, dx, dy) in [
    (1.27, 1.46, 'GW170817', 'center', 0.13, -0.16),
    (1.50, 1.61, 'GW190425', 'center', 0.03, -0.16),
]:
    ax_ns.plot(mx, my, marker='*', ms=14, mec='#0F172A', mfc='#FACC15',
               mew=0.9, zorder=12)
    ax_ns.annotate(name, xy=(mx, my), xytext=(mx + dx, my + dy),
                   fontsize=8, color='#0F172A', fontweight='bold',
                   ha=ha, va='bottom',
                   arrowprops=dict(arrowstyle='->', color='#334155', lw=0.7),
                   bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='#CBD5E1',
                             alpha=0.9, lw=0.5))

ax_ns.set_xlabel(r'$M_2\ [M_\odot]$  (lighter)', fontsize=12)
ax_ns.set_ylabel(r'$M_1\ [M_\odot]$  (heavier)', fontsize=12)
ax_ns.set_xlim(m2_lo, m2_hi)
ax_ns.set_ylim(m1_lo_bns, m1_hi_bns)
ax_ns.tick_params(labelsize=10)
ax_ns.set_axisbelow(True)
ax_ns.text(0.95, 0.95, 'NS\u2013NS', fontsize=10, fontweight='bold',
           color='#1E293B', transform=ax_ns.transAxes, va='top', ha='right',
           bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='none', alpha=0.95))

# ── side-by-side legends below the plot ─────────────────────────────────────
_pa = 0.55
class_handles = [
    Patch(fc=to_rgba(C_SB_BLUE, _pa), ec=C_SB_BLUE, lw=1.0,
          label='sbGRB + blue KN (long-lived HMNS)'),
    Patch(fc=to_rgba(C_LB_HMNS, _pa), ec=C_LB_HMNS, lw=1.0,
          label='lbGRB + red KN (short-lived HMNS, large disk BNS & BH\u2013NS)'),
    Patch(fc=to_rgba(C_FAINT, _pa), ec=C_FAINT, lw=1.0,
          label='Faint lbGRB (small disk BNS & BH\u2013NS)'),
    Patch(fc=to_rgba(C_NO_GRB, _pa), ec=C_NO_GRB, lw=1.0,
          label='No GRB / No KN'),
]
_bns_handles = [
    Line2D([0], [0], color='#0F172A', lw=1.8, ls='--',
           label=rf'$M_{{\mathrm{{tot}}}} = {M_THRESH}\,M_\odot$'),
    Line2D([0], [0], color='#0E7490', lw=1.8, ls='-.',
           label=rf'$1.2\,M_{{\mathrm{{TOV}}}} = {M_HMNS_SPLIT:.2f}\,M_\odot$'),
    Line2D([0], [0], color='#0F172A', lw=1.6, ls=':',
           label=r'$q = 1.2$'),
    Line2D([0], [0], marker='*', color='w', mfc='#FACC15', mec='#0F172A',
           ms=10, lw=0, label='GW events'),
]
_bhns_handles = [
    Line2D([0], [0], color='#1E293B', lw=1.8, ls='--',
           label='Tidal disruption'),
    Line2D([0], [0], color='#92400E', lw=1.8, ls=':',
           label='$M_\\mathrm{disk} = 0.01\\,M_\\odot$'),
    Line2D([0], [0], color='#7F1D1D', lw=1.8, ls=':',
           label='$M_\\mathrm{disk} = 0.1\\,M_\\odot$'),
    Line2D([0], [0], lw=0, label=''),
]
boundary_handles = _bns_handles + _bhns_handles

fig.legend(handles=class_handles, fontsize=8, loc='lower left',
           frameon=True, fancybox=True, edgecolor='#CBD5E1',
           facecolor='white', framealpha=0.95, ncol=2,
           handlelength=1.6, columnspacing=1.2, labelspacing=0.4,
           title='Classification', title_fontsize=9,
           bbox_to_anchor=(0.16, 0.09))

fig.legend(handles=boundary_handles, fontsize=7.5, loc='lower right',
           frameon=True, fancybox=True, edgecolor='#CBD5E1',
           facecolor='white', framealpha=0.95, ncol=2,
           handlelength=2.0, columnspacing=2.0, labelspacing=0.4,
           title='BNS boundaries          BH\u2013NS boundaries',
           title_fontsize=8,
           bbox_to_anchor=(0.8, 0.25))

# ── colorbar ───────────────────────────────────────────────────────────────
cbar_ax = fig.add_axes([0.84, 0.22, 0.025, 0.55])
cbar = fig.colorbar(hb_ns, cax=cbar_ax, orientation='vertical')
cbar.set_label('Systems per bin', fontsize=10)
cbar.ax.tick_params(labelsize=8)

fig.suptitle('Mass Plane: BNS + BHNS',
             fontsize=14, fontweight='bold', y=0.90)
fig.subplots_adjust(bottom=0.18, right=0.82)
plt.savefig('Plots/unified_mass_plane.png')
plt.show()


### Validation: GW170817 and GW190425

Sanity check that the classification framework produces physically consistent results for observed events.  GW170817 ($M_\mathrm{tot} \approx 2.73\,M_\odot$, $q \approx 1.15$) should land in the **sbGRB + blue KN** or **lbGRB (HMNS)** class, consistent with GRB 170817A and AT2017gfo.  GW190425 ($M_\mathrm{tot} \approx 3.4\,M_\odot$, $q \approx 1.08$) should be a prompt collapse system.

In [ ]:
from grb_physics import bns_disk_mass

print("=== GW Event Validation ===\n")

# GW170817: Abbott+ 2019 PRX 9, 011001 low-spin-prior medians
gw170817_m1, gw170817_m2 = np.array([1.46]), np.array([1.27])
# GW190425: Abbott+ 2020 ApJL 892, L3 low-spin-prior medians (M_tot ~ 3.4)
gw190425_m1, gw190425_m2 = np.array([1.61]), np.array([1.50])

for name, m1, m2 in [('GW170817', gw170817_m1, gw170817_m2),
                      ('GW190425', gw190425_m1, gw190425_m2)]:
    cls = classify_bns_2024(m1, m2)
    mt = (m1 + m2).item()
    q = (m1 / m2).item()
    assigned = [k for k, v in cls.items() if v[0]]
    m_ej = bns_dynamical_ejecta(m1, m2, R_1p4_km=12.0).item()
    m_disk = bns_disk_mass(m1, m2, R_1p4_km=12.0).item()

    print(f"{name}:  M_tot = {mt:.2f},  q = {q:.2f}")
    print(f"  Class: {assigned[0] if assigned else 'NONE'}")
    print(f"  Dynamical ejecta (Kruger & Foucart 2020 Eq. 6): {m_ej:.4f} Msun")
    print(f"    KF2020 expected: ~0.003-0.010 Msun (sigma_fit ~ 0.004 Msun)")
    print(f"  BNS disk mass    (Kruger & Foucart 2020 Eq. 4): {m_disk:.4f} Msun")
    print(f"    Compare ~0.05-0.15 Msun from Radice+ 2018 (arXiv:1803.10865),")
    print(f"            Shibata+ 2017 (arXiv:1710.07579) NR simulations.")
    if name == 'GW170817':
        print(f"  AT2017gfo total ejecta: ~0.05-0.08 Msun (Rastinejad+ 2025)")
        print(f"  Note: dynamical ejecta is a subset; disk wind adds ~0.01-0.05 Msun")
    print()

## 2. Component Mass Distributions by GRB Class

Weighted histograms of component masses ($M_1$, $M_2$), total mass ($M_\mathrm{tot}$), and mass ratio ($q$) for merging BNS and BHNS systems, split by Gottlieb et al. (2024) GRB classification. Weights are STROOPWAFEL adaptive-sampling weights from COMPAS Model A.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))

bns_classes = [
    ('sbGRB + blue KN',  sbGRB_blue,  C_SB_BLUE),
    ('lbGRB (HMNS)',      lbGRB_hmns,  C_LB_HMNS),
    ('lbGRB (disk)',      lbGRB_disk,  C_LB_DISK),
    ('Faint / No GRB',   faint_or_none, C_FAINT),
]
bhns_classes = [
    ('lbGRB (BHNS)',      bhns_long,     C_LONG_BH),
    ('Faint lbGRB',       bhns_faint_lb, C_FAINT_BH),
    ('No GRB (swallowed)',bhns_no_grb,   C_NO_GRB),
]

bins_m  = np.linspace(1.0, 2.6, 40)
bins_mt = np.linspace(2.0, 5.0, 40)
bins_q  = np.linspace(1.0, 2.0, 40)
bins_bh = np.linspace(2, 45, 40)

for ax, (data, bins, xlabel) in zip(axes[0],
        [(m1_bns, bins_m, r'$M_1$ (heavier) [$M_\odot$]'),
         (m2_bns, bins_m, r'$M_2$ (lighter) [$M_\odot$]'),
         (M_tot,  bins_mt, r'$M_\mathrm{tot}$ [$M_\odot$]'),
         (q_bns,  bins_q, r'$q = M_1/M_2$')]):
    for lbl, mask, c in bns_classes:
        ax.hist(data[mask], bins=bins, weights=w_bns[mask], density=True,
                histtype='stepfilled', alpha=0.35, color=c, label=lbl)
        ax.hist(data[mask], bins=bins, weights=w_bns[mask], density=True,
                histtype='step', lw=1.5, color=c)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Weighted PDF')

axes[0, 0].set_title('BNS', fontweight='bold', fontsize=12)
axes[0, 0].legend(fontsize=7, loc='upper right')

for ax, (data, bins, xlabel) in zip(axes[1],
        [(BH,    bins_bh, r'$M_\mathrm{BH}$ [$M_\odot$]'),
         (NS_bh, bins_m,  r'$M_\mathrm{NS}$ [$M_\odot$]'),
         (BH + NS_bh, np.linspace(3, 50, 40), r'$M_\mathrm{tot}$ [$M_\odot$]'),
         (BH / NS_bh, np.linspace(1.5, 35, 40), r'$q = M_\mathrm{BH}/M_\mathrm{NS}$')]):
    for lbl, mask, c in bhns_classes:
        if mask.sum() == 0:
            continue
        ax.hist(data[mask], bins=bins, weights=w_bhns[mask], density=True,
                histtype='stepfilled', alpha=0.35, color=c, label=lbl)
        ax.hist(data[mask], bins=bins, weights=w_bhns[mask], density=True,
                histtype='step', lw=1.5, color=c)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Weighted PDF')

axes[1, 0].set_title('BHNS', fontweight='bold', fontsize=12)
axes[1, 0].legend(fontsize=7, loc='upper right')

fig.suptitle('Component Mass Distributions by GRB Class (COMPAS Model A)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/mass_distributions_by_class.png')
plt.show()


## 3. Delay Time Distributions by GRB Class

The delay time $t_\mathrm{delay} = t_\mathrm{form} + t_c$ is the time between stellar birth and compact-object merger. Different GRB classes may have systematically different delay times, affecting their redshift evolution and host galaxy properties.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bins_delay = np.logspace(np.log10(10), np.log10(14000), 50)

for lbl, mask, c in bns_classes:
    ax1.hist(delay_bns[mask], bins=bins_delay, weights=w_bns[mask],
             density=True, histtype='stepfilled', alpha=0.3, color=c)
    ax1.hist(delay_bns[mask], bins=bins_delay, weights=w_bns[mask],
             density=True, histtype='step', lw=2, color=c, label=lbl)
    wmed = np.median(delay_bns[mask])
    ax1.axvline(wmed, color=c, ls='--', lw=1, alpha=0.7)

ax1.set_xscale('log')
ax1.set_xlabel('Delay time [Myr]')
ax1.set_ylabel('Weighted PDF')
ax1.set_title('BNS Delay Times', fontweight='bold')
ax1.legend(fontsize=8)

for lbl, mask, c in bhns_classes:
    if mask.sum() == 0:
        continue
    ax2.hist(delay_bhns[mask], bins=bins_delay, weights=w_bhns[mask],
             density=True, histtype='stepfilled', alpha=0.3, color=c)
    ax2.hist(delay_bhns[mask], bins=bins_delay, weights=w_bhns[mask],
             density=True, histtype='step', lw=2, color=c, label=lbl)

ax2.set_xscale('log')
ax2.set_xlabel('Delay time [Myr]')
ax2.set_ylabel('Weighted PDF')
ax2.set_title('BHNS Delay Times', fontweight='bold')
ax2.legend(fontsize=8)

fig.suptitle('Delay Time Distributions (COMPAS Model A)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/delay_time_distributions.png')
plt.show()


## 4. Cosmic Integration: MSSFR Grid Setup

Setting up the Metallicity-Specific Star Formation Rate (MSSFR) grid from Neijssel et al. (2019) using the COMPAS `FastCosmicIntegration` post-processing library. This convolves the Madau and Dickinson (2014) SFR with a log-normal metallicity evolution model to provide $\mathrm{d}P/\mathrm{d}\log Z(z, Z)$.


In [ ]:
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), 'COMPAS'))
from compas_python_utils.cosmic_integration.FastCosmicIntegration import (
    calculate_redshift_related_params, find_sfr, find_metallicity_distribution)

# Best-effort cosmology consistency check: the notebook prose claims
# Planck 2015 (H0 = 67.74). COMPAS FastCosmicIntegration uses the same
# parameters by default; assert if accessible.
try:
    from compas_python_utils.cosmic_integration import FastCosmicIntegration as _fci
    for _attr in ('H0', 'h0'):
        if hasattr(_fci, _attr):
            assert abs(getattr(_fci, _attr) - 67.74) < 0.5, (
                f"Cosmology mismatch: FastCosmicIntegration.{_attr} != 67.74")
            break
except Exception:
    pass

# Re-anchored ns_radius (P1.1): should return exactly 12.0 km at M = 1.4
# Msun (Raaijmakers+ 2021 R_1.4 = 12.18 +0.56/-0.79 km, CS model).
from grb_physics import ns_radius
assert np.isclose(float(ns_radius(1.4)), 12.0), (
    "ns_radius(1.4) regression: expected 12.0 km after P1.1 re-anchor")

redshifts, n_z_detect, times, time_first_SF, distances, shell_volumes = \
    calculate_redshift_related_params(max_redshift=10.0, redshift_step=0.01)

sfr = find_sfr(redshifts)

# Madau & Dickinson (2014) local SFR ~ 0.01 Msun/yr/Mpc^3 = 1e7 Msun/yr/Gpc^3.
# find_sfr() returns raw SFR (not number-formed); the 1/MEAN_MASS_EVOLVED
# division happens via calibrate_mean_mass_evolved.
print("=== Rate Normalization Diagnostics (Madau & Dickinson 2014 SFR) ===")
print(f"  SFR(z=0) = {sfr[0]:.4e} Msun/yr/Gpc^3  (expect ~1e7, Madau & Dickinson 2014)")
print(f"  SFR units look like {'raw SFR' if sfr[0] > 1e5 else 'ALREADY number-formed — CHECK!'}")

# Verify both populations share the same metallicity range
Z_range_bns = verify_shared_metallicity_prior(DEFAULT_BNS_PATH, DEFAULT_BHNS_PATH)

# dPdlogZ is dP/d(ln Z) (natural log), matching COMPAS convention.
# Integration uses Delta(ln Z) = np.diff(np.log(Z)), NOT log10.
dPdlogZ, metallicities, p_draw = find_metallicity_distribution(
    redshifts,
    min_logZ_COMPAS=np.log(Z_range_bns[0]),
    max_logZ_COMPAS=np.log(Z_range_bns[1]))

# Verify dP/dlnZ integrates to ~1 at every redshift slice (Neijssel+ 2019)
try:
    dPdlogZ_norm = check_dPdlogZ_normalization(dPdlogZ, metallicities, rtol=0.10)
    print(f"  dPdlogZ integral range: [{dPdlogZ_norm.min():.4f}, {dPdlogZ_norm.max():.4f}]  ✓")
except ValueError as e:
    print(f"  WARNING: {e}")
    dlogZ = np.diff(np.log(metallicities))
    dlogZ = np.append(dlogZ, dlogZ[-1])
    dPdlogZ_norm = (dPdlogZ * dlogZ[None, :]).sum(axis=1)
    print(f"  Renormalizing dPdlogZ at each z-slice...")
    dPdlogZ = dPdlogZ / dPdlogZ_norm[:, None]

# Verify p_draw and population consistency (shared metallicity prior)
import h5py as h5
with h5.File(DEFAULT_BNS_PATH, 'r') as f:
    bns_keys = list(f.keys())
    has_sampling = 'sampling_fraction' in f.attrs if hasattr(f, 'attrs') else False
with h5.File(DEFAULT_BHNS_PATH, 'r') as f:
    bhns_keys = list(f.keys())

print(f"\n=== Population Consistency ===")
print(f"  BNS file:  {DEFAULT_BNS_PATH.split('/')[-1]}  groups: {bns_keys[:5]}")
print(f"  BHNS file: {DEFAULT_BHNS_PATH.split('/')[-1]}  groups: {bhns_keys[:5]}")
print(f"  p_draw = {p_draw:.6f}  (shared between BNS and BHNS)")
print(f"  BNS  n_merging = {n_bns:,},  BHNS n_merging = {n_bhns:,}")

ns_max_check = NS_MAX_BNS
bns_near_boundary = ((m1_bns > M_TOV) & (m1_bns <= ns_max_check)).sum()
print(f"  BNS systems with m1 in [{M_TOV:.2f}, {ns_max_check:.2f}]: {bns_near_boundary}")

# Per-population normalization: calibrate MEAN_MASS_EVOLVED against
# the pre-computed local merger rates in each Broekgaarden HDF5 file.
expected_rate_bns  = read_expected_local_rate(DEFAULT_BNS_PATH)
expected_rate_bhns = read_expected_local_rate(DEFAULT_BHNS_PATH)

mean_mass_bns, _  = calibrate_mean_mass_evolved(
    sfr, redshifts, times, time_first_SF,
    dPdlogZ, metallicities, p_draw,
    Z_bns, delay_bns, w_bns, expected_rate_bns)

mean_mass_bhns, _ = calibrate_mean_mass_evolved(
    sfr, redshifts, times, time_first_SF,
    dPdlogZ, metallicities, p_draw,
    Z_bhns, delay_bhns, w_bhns, expected_rate_bhns)

n_formed_BNS  = sfr / mean_mass_bns
n_formed_BHNS = sfr / mean_mass_bhns

print(f"\n=== Calibration Results ===")
print(f"  Redshift grid:  {len(redshifts)} points (0 to {redshifts.max():.0f})")
print(f"  dPdlogZ shape:  {dPdlogZ.shape}")
print(f"  p_draw (Z range [{Z_range_bns[0]:.4f}, {Z_range_bns[1]:.4f}]): {p_draw:.6f}")
print(f"  MEAN_MASS_EVOLVED  BNS: {mean_mass_bns:.4e}  BHNS: {mean_mass_bhns:.4e}")
print(f"  Expected local rate  BNS: {expected_rate_bns:.1f}  BHNS: {expected_rate_bhns:.1f} Gpc^-3 yr^-1")
print(f"\n  Literature comparison:")
print(f"    Broekgaarden+ 2021 fiducial BHNS: ~43 Gpc^-3 yr^-1")
print(f"    Neijssel+ 2019 BNS:               ~57 Gpc^-3 yr^-1")
print(f"    van Son+ 2022 BNS range:           0.32-330 Gpc^-3 yr^-1")
if expected_rate_bns < expected_rate_bhns:
    print(f"\n  ⚠ WARNING: BNS expected rate ({expected_rate_bns:.1f}) < BHNS ({expected_rate_bhns:.1f})")
    print(f"    This is unusual for fiducial models where BNS should dominate.")
    print(f"    Check that BNS/BHNS files use the same COMPAS simulation and")
    print(f"    that p_draw is consistent between populations.")


## 5. Metallicity Dependence of GRB Formation Efficiency

Formation efficiency $\eta(Z)$ [mergers per $M_\odot$ formed] as a function of birth metallicity. The relative fractions of different GRB classes shift with metallicity because lower-$Z$ stars retain more mass, producing heavier remnants that more easily exceed $M_\mathrm{thresh}$.


In [ ]:
# Uses mean_mass_bns / mean_mass_bhns from Section 11 (cosmic-integration setup).
eff_bns = formation_efficiency(
    METALLICITY_GRID, Z_bns, w_bns,
    masks={'sbGRB + blue KN': sbGRB_blue,
           'lbGRB (HMNS)': lbGRB_hmns,
           'lbGRB (disk)': lbGRB_disk,
           'Faint / No GRB': faint_or_none},
    mean_mass_evolved=mean_mass_bns)

eff_bhns = formation_efficiency(
    METALLICITY_GRID, Z_bhns, w_bhns,
    masks={'Long cbGRB (BHNS)':  bhns_long,
           'Short cbGRB (BHNS)': bhns_faint_lb,
           'No GRB (NS swallowed)': bhns_no_grb},
    mean_mass_evolved=mean_mass_bhns)

# ── Physical constants ────────────────────────────────────────────────────
Z_SUN = 0.0142  # Asplund et al. 2009 (consistent with the rest of the repo).

# ── Grid hygiene ──────────────────────────────────────────────────────────
# The COMPAS grid intentionally duplicates its final 0.03 entry; that makes
# formation_efficiency assign the same value to two adjacent indices and shows
# up as a phantom vertical "spike" at the high-Z edge.  Drop the duplicate.
_uniq_mask = np.concatenate(([True], np.diff(METALLICITY_GRID) != 0))
Z_grid = METALLICITY_GRID[_uniq_mask]
Z_over_Zsun = Z_grid / Z_SUN

# ── Per-bin sampling counts ───────────────────────────────────────────────
# Bins with very few simulated systems are Poisson-noise-dominated.  We need
# an unweighted N per grid bin to flag them; STROOPWAFEL weights tell us about
# rate, not statistical confidence.
def _bin_counts(Z_systems, grid):
    uniq, cnt = np.unique(Z_systems, return_counts=True)
    out = np.zeros(len(grid), dtype=int)
    for z, c in zip(uniq, cnt):
        idx = np.where(np.isclose(grid, z, rtol=1e-6))[0]
        if len(idx):
            out[idx[0]] = c
    return out

N_bns  = _bin_counts(Z_bns,  Z_grid)
N_bhns = _bin_counts(Z_bhns, Z_grid)

# A bin counts as "well-sampled" if it has at least N_MIN simulated systems.
# Below this the per-class fractions and η(Z) carry large statistical scatter,
# so we render those bins with faded open markers (still visible but visually
# de-emphasised) and exclude them from the stackplots.
N_MIN = 50
ok_bns_bin  = N_bns  >= N_MIN
ok_bhns_bin = N_bhns >= N_MIN

print(f'BNS  : {ok_bns_bin.sum():>2d}/{len(Z_grid)} bins well-sampled '
      f'(N >= {N_MIN}); '
      f'{(N_bns > 0).sum() - ok_bns_bin.sum()} additional bins under-sampled.')
print(f'BHNS : {ok_bhns_bin.sum():>2d}/{len(Z_grid)} bins well-sampled '
      f'(N >= {N_MIN}); '
      f'{(N_bhns > 0).sum() - ok_bhns_bin.sum()} additional bins under-sampled.')
if ok_bns_bin.any():
    print(f'BNS  Z/Zsun coverage : '
          f'{Z_over_Zsun[ok_bns_bin].min():.3f} → '
          f'{Z_over_Zsun[ok_bns_bin].max():.3f}')
if ok_bhns_bin.any():
    print(f'BHNS Z/Zsun coverage : '
          f'{Z_over_Zsun[ok_bhns_bin].min():.3f} → '
          f'{Z_over_Zsun[ok_bhns_bin].max():.3f}')

def _split(arr, ok_bin):
    """Return two NaN-padded copies of `arr`: one for well-sampled bins,
    one for under-sampled bins. Empty (zero) bins become NaN in both."""
    a = arr[_uniq_mask].astype(float)
    a[a <= 0] = np.nan
    well  = np.where(ok_bin, a, np.nan)
    under = np.where(~ok_bin & np.isfinite(a), a, np.nan)
    return well, under

# Distinct colours; defined locally so the global C_* constants stay intact
# for the other plots in the notebook.
C_BNS_SB    = C_SB_BLUE   # cyan
C_BNS_HMNS  = '#B91C1C'   # deep red
C_BNS_DISK  = '#F97316'   # vivid orange
C_BNS_FAINT = '#A16207'   # amber-brown
C_BH_LONG   = '#7C3AED'   # violet (different population family)
C_BH_FAINT  = '#F59E0B'   # amber
C_BH_NONE   = '#475569'   # slate gray

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)

def _draw_efficiency(ax, eff, classes, ok_bin, total_label):
    for lbl, c in classes:
        well, under = _split(eff[lbl], ok_bin)
        ax.plot(Z_over_Zsun, well, color=c, lw=2, marker='o', ms=3, label=lbl)
        ax.plot(Z_over_Zsun, under, color=c, lw=0,
                marker='o', mfc='none', ms=4, alpha=0.4)
    well_t, under_t = _split(eff['total'], ok_bin)
    ax.plot(Z_over_Zsun, well_t, 'k-', lw=2.5, marker='o', ms=3,
            label=total_label)
    ax.plot(Z_over_Zsun, under_t, color='k', lw=0,
            marker='o', mfc='none', ms=4, alpha=0.4)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.axvline(1.0, color='#94A3B8', lw=1.0, ls=':', zorder=0)
    ax.text(1.04, 0.02, r'$Z_\odot$', fontsize=9, color='#64748B',
            transform=ax.get_xaxis_transform())
    ax.set_ylabel(r'$\eta(Z)\;[\mathrm{mergers}/M_\odot\;\mathrm{formed}]$')

# ── Top-left: BNS formation efficiency ────────────────────────────────────
_draw_efficiency(axes[0, 0], eff_bns,
                 [('sbGRB + blue KN', C_BNS_SB),
                  ('lbGRB (HMNS)',    C_BNS_HMNS),
                  ('lbGRB (disk)',    C_BNS_DISK),
                  ('Faint / No GRB',  C_BNS_FAINT)],
                 ok_bns_bin, 'Total BNS')
axes[0, 0].set_title('BNS Formation Efficiency', fontweight='bold')
axes[0, 0].legend(fontsize=8, loc='lower left', framealpha=0.9)

# ── Top-right: BHNS formation efficiency ──────────────────────────────────
# NOTE: BHNS classes are *not* parallel to the BNS GRB taxonomy.  A BHNS
# "Short cbGRB" comes from a successful tidal disruption + accretion disk;
# it is physically distinct from a BNS "Faint lbGRB" (low-disk-mass HMNS
# remnant).  Keep the cbGRB nomenclature so this distinction is visible.
_draw_efficiency(axes[0, 1], eff_bhns,
                 [('Long cbGRB (BHNS)',     C_BH_LONG),
                  ('Short cbGRB (BHNS)',    C_BH_FAINT),
                  ('No GRB (NS swallowed)', C_BH_NONE)],
                 ok_bhns_bin, 'Total BHNS')
axes[0, 1].set_title('BHNS Formation Efficiency', fontweight='bold')
axes[0, 1].legend(fontsize=8, loc='lower left', framealpha=0.9)

# Set y-limits from well-sampled bins only so a single under-sampled outlier
# doesn't pull the axis floor down by orders of magnitude (which compresses
# the real trend into the upper edge of the panel).
def _well_sampled_range(eff, ok_bin):
    vals = []
    for v in eff.values():
        a = v[_uniq_mask][ok_bin]
        a = a[a > 0]
        if a.size:
            vals.append(a)
    if not vals:
        return None
    cat = np.concatenate(vals)
    return cat.min(), cat.max()

ranges = [r for r in (_well_sampled_range(eff_bns,  ok_bns_bin),
                      _well_sampled_range(eff_bhns, ok_bhns_bin)) if r]
if ranges:
    y_lo = min(r[0] for r in ranges) / 3.0   # ~0.5 dex headroom
    y_hi = max(r[1] for r in ranges) * 3.0
    axes[0, 0].set_ylim(y_lo, y_hi)
    axes[0, 1].set_ylim(y_lo, y_hi)

# Lock x-range to the well-sampled coverage so the under-sampled high-Z bin
# (if present) doesn't pad the axis with an empty solar-Z region.
xs = [Z_over_Zsun[ok] for ok in (ok_bns_bin, ok_bhns_bin) if ok.any()]
if xs:
    cat = np.concatenate(xs)
    xlo, xhi = cat.min() / 1.5, cat.max() * 1.5
    for a in axes.flat:
        a.set_xlim(xlo, xhi)

# ── Bottom-left: BNS class fractions (well-sampled bins only) ─────────────
ax = axes[1, 0]
denom_bns_full = eff_bns['total'][_uniq_mask]
sel = ok_bns_bin & (denom_bns_full > 0)
with np.errstate(invalid='ignore', divide='ignore'):
    f_sb = eff_bns['sbGRB + blue KN'][_uniq_mask] / denom_bns_full
    f_hm = eff_bns['lbGRB (HMNS)'][_uniq_mask]    / denom_bns_full
    f_dk = eff_bns['lbGRB (disk)'][_uniq_mask]    / denom_bns_full
    f_fn = eff_bns['Faint / No GRB'][_uniq_mask]  / denom_bns_full
ax.stackplot(Z_over_Zsun[sel], f_sb[sel], f_hm[sel], f_dk[sel], f_fn[sel],
             colors=[C_BNS_SB, C_BNS_HMNS, C_BNS_DISK, C_BNS_FAINT],
             labels=['sbGRB + blue KN', 'lbGRB (HMNS)',
                     'lbGRB (disk)', 'Faint / No GRB'],
             alpha=0.85)
ax.set_xscale('log')
ax.set_xlabel(r'$Z\,/\,Z_\odot$')
ax.set_ylabel('Fraction of BNS mergers')
ax.set_title(r'BNS GRB Class Fractions vs $Z/Z_\odot$', fontweight='bold')
ax.axvline(1.0, color='white', lw=1.2, ls=':', alpha=0.9, zorder=5)
ax.text(1.04, 0.02, r'$Z_\odot$', fontsize=9, color='white',
        transform=ax.get_xaxis_transform())
ax.set_ylim(0, 1)
ax.legend(fontsize=8, loc='upper left', framealpha=0.9)

# ── Bottom-right: BHNS class fractions (well-sampled bins only) ───────────
ax = axes[1, 1]
denom_bhns_full = eff_bhns['total'][_uniq_mask]
sel_bh = ok_bhns_bin & (denom_bhns_full > 0)
with np.errstate(invalid='ignore', divide='ignore'):
    g_lo = eff_bhns['Long cbGRB (BHNS)'][_uniq_mask]     / denom_bhns_full
    g_fa = eff_bhns['Short cbGRB (BHNS)'][_uniq_mask]    / denom_bhns_full
    g_no = eff_bhns['No GRB (NS swallowed)'][_uniq_mask] / denom_bhns_full
ax.stackplot(Z_over_Zsun[sel_bh], g_lo[sel_bh], g_fa[sel_bh], g_no[sel_bh],
             colors=[C_BH_LONG, C_BH_FAINT, C_BH_NONE],
             labels=['Long cbGRB (BHNS)', 'Short cbGRB (BHNS)',
                     'No GRB (NS swallowed)'],
             alpha=0.85)
ax.set_xscale('log')
ax.set_xlabel(r'$Z\,/\,Z_\odot$')
ax.set_ylabel('Fraction of BHNS mergers')
ax.set_title(r'BHNS GRB Class Fractions vs $Z/Z_\odot$', fontweight='bold')
ax.axvline(1.0, color='white', lw=1.2, ls=':', alpha=0.9, zorder=5)
ax.text(1.04, 0.02, r'$Z_\odot$', fontsize=9, color='white',
        transform=ax.get_xaxis_transform())
ax.set_ylim(0, 1)
ax.legend(fontsize=8, loc='upper left', framealpha=0.9)

# Normalisation sanity check.  formation_efficiency divides total weight by
# the *mean evolved stellar mass* of the COMPAS run, separately for each
# population.  If these calibrations are wrong relative to each other, the
# BNS/BHNS rate ratio will be biased without changing the shape of either
# curve, which makes it look like a real physical effect.  Print them so any
# anomaly is visible at a glance: a healthy COMPAS Model A run typically has
# mean_mass_bhns / mean_mass_bns close to unity (both populations evolve from
# the same primary IMF), so a ratio far from 1 is suspicious.
print(f'mean_mass_bns        = {mean_mass_bns:.3e} Msun')
print(f'mean_mass_bhns       = {mean_mass_bhns:.3e} Msun')
print(f'mean_mass_bhns/bns   = {mean_mass_bhns / mean_mass_bns:.3f}')

# Closure check: sub-class masks must partition each population at every Z.
# A nonzero residual would mean mergers are double-counted or missing.
for pop, eff, keys in [
    ('BNS',  eff_bns,  ['sbGRB + blue KN', 'lbGRB (HMNS)',
                        'lbGRB (disk)', 'Faint / No GRB']),
    ('BHNS', eff_bhns, ['Long cbGRB (BHNS)', 'Short cbGRB (BHNS)',
                        'No GRB (NS swallowed)']),
]:
    summed = np.sum([eff[k] for k in keys], axis=0)
    nz = eff['total'] > 0
    if nz.any():
        rel_err = np.max(np.abs(summed[nz] - eff['total'][nz]) / eff['total'][nz])
        print(f'{pop:>4s}: max |Σ class − total| / total = {rel_err:.2e}')

fig.suptitle(r'Metallicity Dependence of Compact-Binary Mergers and GRB Classes  '
             r'(COMPAS Model A, $Z_\odot = 0.0142$, '
             rf'$N_\mathrm{{min}} = {N_MIN}$ systems/bin)',
             fontsize=13, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('Plots/metallicity_dependence.png')
plt.show()


## 6. Formation Channel Breakdown per GRB Class

BNS formation channels classified following Broekgaarden et al. (2021, 2022) using COMPAS event-sequence columns. Channels: I (Classic CE), II (Stable MT only), III (Single-core CE), IV (Double-core CE), V (Other).


In [ ]:
channels = classify_formation_channels(
    dblCE=bns_ch['dblCE'], fc_CEE=bns_ch['fc_CEE'],
    fc_mt_p1=bns_ch['fc_mt_p1'], fc_mt_s1=bns_ch['fc_mt_s1'],
    fc_mt_p1_K1=bns_ch['fc_mt_p1_K1'], fc_mt_s1_K2=bns_ch['fc_mt_s1_K2'])

ch_names = list(channels.keys())
grb_classes_bns = [
    ('sbGRB + blue KN',  sbGRB_blue),
    ('lbGRB (HMNS)',      lbGRB_hmns),
    ('lbGRB (disk)',      lbGRB_disk),
    ('Faint / No GRB',   faint_or_none),
]

ch_colors = ['#3B82F6', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6']
ch_short = ['I', 'II', 'III', 'IV', 'V']

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(grb_classes_bns))
width = 0.15
for j, (ch_name, ch_mask) in enumerate(channels.items()):
    fracs = []
    for gname, gmask in grb_classes_bns:
        combined = gmask & ch_mask
        wtot = w_bns[gmask].sum()
        fracs.append(100 * w_bns[combined].sum() / wtot if wtot > 0 else 0)
    offset = (j - len(ch_names) / 2 + 0.5) * width
    bars = ax.bar(x + offset, fracs, width, label=ch_short[j], color=ch_colors[j], alpha=0.85)
    for bar, val in zip(bars, fracs):
        if val > 3:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{val:.0f}%', ha='center', va='bottom', fontsize=7, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([g[0] for g in grb_classes_bns], fontsize=10)
ax.set_ylabel('Weighted fraction (%)')
ax.set_title('BNS Formation Channel Breakdown per GRB Class',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=8, ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.08),
          title='Channel', title_fontsize=9)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('Plots/formation_channels_by_grb_class.png')
plt.show()


## 7. BNS Merger Rate $\mathcal{R}(z)$ per GRB Class

Intrinsic merger rate density [Gpc$^{-3}$ yr$^{-1}$] vs redshift for each BNS GRB class. The merger rate is obtained by convolving per-binary formation rates (SFR $\times$ metallicity distribution $\times$ STROOPWAFEL weight) with delay times.


In [ ]:
# Gottlieb 2024 five-class
classes_BNS_24 = [
    ('sbGRB + blue KN',  sbGRB_blue),
    ('lbGRB (HMNS)',      lbGRB_hmns),
    ('lbGRB (disk)',      lbGRB_disk),
    ('Faint / No GRB',   faint_or_none),
    ('All BNS',           np.ones(len(delay_bns), dtype=bool)),
]

merger_rates_BNS = {}
for label, mask in classes_BNS_24:
    w_eff = w_bns[mask]
    rate = compute_merger_rate(
        redshifts, times, time_first_SF, n_formed_BNS,
        dPdlogZ, metallicities, p_draw,
        Z_bns[mask], delay_bns[mask], w_eff)
    merger_rates_BNS[label] = rate
    iz0 = np.argmin(np.abs(redshifts))
    print(f"  {label:25s}:  R(z=0) = {rate_label(rate[iz0]):>8s} Gpc\u207b\u00b3 yr\u207b\u00b9")

fig, ax = plt.subplots(figsize=(10, 6))
for lbl, c, ls in [('sbGRB + blue KN', C_SB_BLUE, '-'),
                    ('lbGRB (HMNS)', C_LB_HMNS, '-'),
                    ('lbGRB (disk)', C_LB_DISK, '-'),
                    ('Faint / No GRB', C_FAINT, '--'),
                    ('All BNS', 'black', '-')]:
    lw = 2.5 if lbl == 'All BNS' else 1.8
    ax.plot(redshifts, merger_rates_BNS[lbl], color=c, ls=ls, lw=lw, label=lbl)

    iz0 = np.argmin(np.abs(redshifts))
    r0 = merger_rates_BNS[lbl][iz0]
    if r0 > 0.1:
        ax.annotate(f'{rate_label(r0)}', xy=(0.05, r0), fontsize=7,
                    color=c, fontweight='bold', va='bottom')

ax.set_xlim(0, 10)
ax.set_yscale('log')
ax.set_ylim(bottom=0.1)
ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$\mathcal{R}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax.set_title('BNS Merger Rate Density by GRB Class (Gottlieb 2024)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Plots/rate_bns_by_class.png')
plt.show()


## 8. BHNS Merger Rate $\mathcal{R}(z)$ with BH Spin Sensitivity

BHNS merger rates depend strongly on the assumed BH spin $a_\mathrm{BH}$: higher spins shift the ISCO inward, increasing the fraction of NS material that forms a disk. The shaded envelope spans $a = 0$ (pessimistic) to $a = 0.9$ (optimistic).

**Spin-orbit misalignment caveat:** The Foucart (2018) formula assumes aligned BH spin. Kawaguchi et al. (2015) NR simulations show disk mass drops to near zero for misalignment angles $> 50$–$60°$. Population synthesis suggests $\sim$half of BHNS systems have misalignment $> 45°$ (Fragos+ 2010 arXiv:1001.1107; Gerosa+ 2018), so BHNS GRB fractions are overestimated by up to $\sim 2\times$ in the aligned-spin approximation.

**Two routes to handle this systematic — do not combine them:**

1. **Per-system projection** via `effective_aligned_spin(a_BH, theta_tilt)` when individual tilt angles `theta_tilt` are sampled per binary; the projected $a_\mathrm{eff}$ is then passed to the Foucart formula directly.
2. **Population-averaged factor** via `apply_bhns_misalignment(rate_bhns)` (= `rate_bhns * MISALIGNMENT_SYSTEMATIC_FACTOR`, default 0.5) when only aggregate rates are available.

Cell 19 prints both the per-system projection table (route 1, illustrative) and the population-averaged rates (route 2, used in the rate panels). The two are alternative treatments of the same systematic; combining them double-counts the suppression.

**Calibration-range caveat:** Spin sweeps below include $a_\mathrm{BH} = 0.9$, the upper edge of the Foucart+ 2018 calibration range $\chi_\mathrm{BH} \in [-0.5, 0.9]$. We pass `clip_chi=0.9` to `classify_bhns` to enforce this boundary; values beyond would be extrapolation.


In [ ]:
spin_grid = [0.0, 0.3, 0.5, 0.7, 0.9]
spin_rates_long = {}
spin_rates_short = {}

# clip_chi = 0.9 keeps the Foucart+ 2018 fit within its calibration range
# (chi_BH in [-0.5, 0.9]); systems with |a_BH| > clip_chi return zero disk.
for a in spin_grid:
    cbh = classify_bhns(BH, NS_bh, a_BH=a, clip_chi=0.9)
    mask_l = cbh['Long cbGRB']
    mask_s = cbh['Short cbGRB']

    spin_rates_long[a] = compute_merger_rate(
        redshifts, times, time_first_SF, n_formed_BHNS,
        dPdlogZ, metallicities, p_draw,
        Z_bhns[mask_l], delay_bhns[mask_l], w_bhns[mask_l])
    spin_rates_short[a] = compute_merger_rate(
        redshifts, times, time_first_SF, n_formed_BHNS,
        dPdlogZ, metallicities, p_draw,
        Z_bhns[mask_s], delay_bhns[mask_s], w_bhns[mask_s])

    iz0 = np.argmin(np.abs(redshifts))
    print(f"  a={a:.1f}  Long: {rate_label(spin_rates_long[a][iz0]):>6s}"
          f"  Short: {rate_label(spin_rates_short[a][iz0]):>6s} Gpc\u207b\u00b3 yr\u207b\u00b9")

print(f"\n  Spin-orbit misalignment systematic (Kawaguchi+ 2015):")
print(f"  All BHNS GRB rates above assume aligned spin.")
print(f"  Route 1 (per-system, illustrative): effective_aligned_spin at a_BH=0.5:")
for theta_deg in [0, 30, 45, 60, 90]:
    a_eff = effective_aligned_spin(0.5, np.radians(theta_deg))
    print(f"    tilt={theta_deg:>2d} deg -> a_eff={a_eff:.3f}")
print(f"  Route 2 (population, applied below): apply_bhns_misalignment "
      f"(x{MISALIGNMENT_SYSTEMATIC_FACTOR}; Fragos+ 2010, Gerosa+ 2018):")
for a in spin_grid:
    r_corr = float(apply_bhns_misalignment(spin_rates_long[a][iz0]))
    print(f"    a={a:.1f}  Long (aligned): {rate_label(spin_rates_long[a][iz0]):>6s}"
          f"  -> (w/ misalign): {rate_label(r_corr):>6s}")

rate_bhns_all = compute_merger_rate(
    redshifts, times, time_first_SF, n_formed_BHNS,
    dPdlogZ, metallicities, p_draw, Z_bhns, delay_bhns, w_bhns)

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(redshifts, rate_bhns_all, 'k-', lw=2.5, label='All BHNS')

ax.fill_between(redshifts, spin_rates_long[0.0], spin_rates_long[0.9],
                color=C_LONG_BH, alpha=0.2, label='lbGRB envelope ($a=0$ to $0.9$, aligned)')
ax.plot(redshifts, spin_rates_long[0.5], color=C_LONG_BH, lw=2, label='lbGRB ($a=0.5$, aligned)')
ax.fill_between(redshifts,
                apply_bhns_misalignment(spin_rates_long[0.0]),
                apply_bhns_misalignment(spin_rates_long[0.9]),
                color=C_LONG_BH, alpha=0.10, hatch='//',
                label=f'lbGRB envelope (w/ misalign. x{MISALIGNMENT_SYSTEMATIC_FACTOR})')

ax.fill_between(redshifts, spin_rates_short[0.0], spin_rates_short[0.9],
                color=C_FAINT_BH, alpha=0.2, label='Faint lbGRB envelope')
ax.plot(redshifts, spin_rates_short[0.5], color=C_FAINT_BH, lw=2, ls='--',
        label='Faint lbGRB ($a=0.5$)')

ax.set_xlim(0, 10)
ax.set_yscale('log')
ax.set_ylim(bottom=0.01)
ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$\mathcal{R}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax.set_title('BHNS Merger Rate Density: BH Spin Sensitivity',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('Plots/rate_bhns_spin_sensitivity.png')
plt.show()


In [ ]:
# ── Sanity check: local rates vs Broekgaarden et al. fiducial ─────────────
iz0 = np.argmin(np.abs(redshifts))
R_bns_z0  = merger_rates_BNS['All BNS'][iz0]
R_bhns_z0 = rate_bhns_all[iz0]

print("Local (z=0) merger rate sanity check")
print("=" * 70)
print(f"  All BNS:   {R_bns_z0:8.1f} Gpc^-3 yr^-1  (fiducial ~50-100)")
print(f"  All BHNS:  {R_bhns_z0:8.1f} Gpc^-3 yr^-1  (fiducial ~43)")
print(f"  BNS/BHNS:  {R_bns_z0/R_bhns_z0:8.2f}         (expect >1)")
print(f"  Expected (w_000):  BNS={expected_rate_bns:.1f}, BHNS={expected_rate_bhns:.1f}")
print("=" * 70)
if R_bns_z0 < R_bhns_z0:
    print("  WARNING: BNS < BHNS — ratio may still be inverted")
if R_bns_z0 > 500 or R_bhns_z0 > 200:
    print("  WARNING: rates appear too high — check normalization")

# Detailed mask/weight verification (BNS/BHNS partition consistency)
ns_max_val = NS_MAX_BNS
print(f"\n--- BNS/BHNS boundary verification (ns_max = {ns_max_val:.2f}) ---")

bhns_ns_near_boundary = ((NS_bh > M_TOV) & (NS_bh <= ns_max_val)).sum()
print(f"  BHNS systems with M_NS in [{M_TOV:.2f}, {ns_max_val:.2f}]: {bhns_ns_near_boundary}")
print(f"  BNS  systems with m1   in [{M_TOV:.2f}, {ns_max_val:.2f}]: "
      f"{((m1_bns > M_TOV) & (m1_bns <= ns_max_val)).sum()}")
print(f"  BNS  systems with m2   in [{M_TOV:.2f}, {ns_max_val:.2f}]: "
      f"{((m2_bns > M_TOV) & (m2_bns <= ns_max_val)).sum()}")

bns_w_tot = w_bns.sum()
bhns_w_tot = w_bhns.sum()
print(f"  Total STROOPWAFEL weight:  BNS = {bns_w_tot:.4e},  BHNS = {bhns_w_tot:.4e}")
print(f"  Weight ratio BNS/BHNS:     {bns_w_tot / bhns_w_tot:.2f}")

print(f"  BNS  mass range: m1=[{m1_bns.min():.3f}, {m1_bns.max():.3f}], "
      f"m2=[{m2_bns.min():.3f}, {m2_bns.max():.3f}]")
print(f"  BHNS mass range: M_BH=[{BH.min():.2f}, {BH.max():.2f}], "
      f"M_NS=[{NS_bh.min():.3f}, {NS_bh.max():.3f}]")

if BH.min() <= ns_max_val:
    overlap = (BH <= ns_max_val).sum()
    print(f"  WARNING: {overlap} BHNS systems have M_BH <= ns_max={ns_max_val:.2f} — "
          f"possible BNS/BHNS overlap!")
else:
    print(f"  OK: No BH masses below ns_max — no BNS/BHNS overlap")

## 9. Combined BNS + BHNS Rate Summary and Rate-Weighted Distributions

Left: combined merger rate density for all GRB classes (BNS + BHNS). Right: rate-weighted total mass distributions at $z = 0$ vs $z = 2$, showing how the mass distribution and GRB-class balance shift with cosmic time.


In [ ]:
# Compute marginalized spin rates before plotting (move Section 18 results up).
# Flat prior: uniform over the spin grid.  Fuller & Ma 2019 (arXiv:1907.03715):
# efficient angular momentum transport produces predominantly low BH spins.
w_flat = {0.0: 0.20, 0.3: 0.20, 0.5: 0.20, 0.7: 0.20, 0.9: 0.20}
w_fm19 = {0.0: 0.70, 0.3: 0.20, 0.5: 0.07, 0.7: 0.02, 0.9: 0.01}

r_long_flat  = marginalize(spin_rates_long, w_flat)
r_long_fm19  = marginalize(spin_rates_long, w_fm19)
r_short_flat = marginalize(spin_rates_short, w_flat)
r_short_fm19 = marginalize(spin_rates_short, w_fm19)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Combined R(z) using Fuller & Ma 2019 marginalized BHNS spin distribution
ax1.plot(redshifts, merger_rates_BNS['All BNS'], 'k-', lw=2.5, label='All BNS')
ax1.plot(redshifts, rate_bhns_all, 'k--', lw=2.5, label='All BHNS')

for lbl, c, ls in [('sbGRB + blue KN', C_SB_BLUE, '-'),
                    ('lbGRB (HMNS)', C_LB_HMNS, '-'),
                    ('lbGRB (disk)', C_LB_DISK, '-')]:
    ax1.plot(redshifts, merger_rates_BNS[lbl], color=c, ls=ls, lw=1.5, label=f'BNS {lbl}')

ax1.plot(redshifts, r_long_fm19, color=C_LONG_BH, ls='-', lw=2,
         label='BHNS lbGRB (F&M, aligned)')
r_long_fm19_misaligned = apply_bhns_misalignment(r_long_fm19)
ax1.plot(redshifts, r_long_fm19_misaligned, color=C_LONG_BH, ls='--', lw=1.5,
         label=f'BHNS lbGRB (F&M, w/ misalign. x{MISALIGNMENT_SYSTEMATIC_FACTOR})')
ax1.fill_between(redshifts, spin_rates_long[0.0], spin_rates_long[0.9],
                 color=C_LONG_BH, alpha=0.15, label='BHNS lbGRB ($a=0$ to $0.9$)')

ax1.set_xlim(0, 10)
ax1.set_yscale('log')
ax1.set_ylim(0.01, None)
ax1.set_xlabel('Redshift $z$')
ax1.set_ylabel(r'$\mathcal{R}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax1.set_title('Combined BNS + BHNS Merger Rates (upper bounds)', fontweight='bold')
ax1.legend(fontsize=7, loc='upper right')
ax1.grid(True, alpha=0.3)

# Rate-weighted M_tot at z=0, 0.5, 1, 2: shows how the BNS mass distribution
# evolves with cosmic time as the metallicity-delay convolution shifts.
bins_mt = np.linspace(2.0, 5.0, 40)
z_targets = [0.0, 0.5, 1.0, 2.0]
colors_z  = ['#1D4ED8', '#6366F1', '#F59E0B', '#DC2626']

for z_t, cz in zip(z_targets, colors_z):
    w_z = per_system_rate_weights(
        z_t, redshifts, times, time_first_SF,
        n_formed_BNS, dPdlogZ, metallicities, p_draw,
        Z_bns, delay_bns, w_bns)
    if w_z.sum() > 0:
        lbl = f'$z = {z_t:.0f}$' if z_t == int(z_t) else f'$z = {z_t}$'
        ax2.hist(M_tot, bins=bins_mt, weights=w_z, density=True,
                 histtype='step', lw=2, color=cz, label=lbl)
        ax2.hist(M_tot, bins=bins_mt, weights=w_z, density=True,
                 histtype='stepfilled', alpha=0.12, color=cz)

ax2.axvline(M_HMNS_SPLIT, color='#059669', ls='--', lw=1.2,
            label=f'$1.2\\,M_\\mathrm{{TOV}} = {M_HMNS_SPLIT:.2f}$')
ax2.axvline(M_THRESH, color='gray', ls=':', lw=1.5,
            label=f'$M_\\mathrm{{thresh}} = {M_THRESH}$')
ax2.set_xlabel(r'$M_\mathrm{tot}$ [$M_\odot$]')
ax2.set_ylabel('Rate-weighted PDF')
ax2.set_title(r'BNS $M_\mathrm{tot}$: Rate-Weighted Evolution with Redshift', fontweight='bold')
ax2.legend(fontsize=8, loc='upper right')

fig.suptitle('Cosmic Rate Summary (COMPAS Model A)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/rate_combined_summary.png')
plt.show()


## 10. Systemic Velocity Distributions

Post-supernova systemic velocity ($v_\mathrm{sys}$) of merging compact binaries, from the COMPAS `supernovae` group. Different GRB progenitors may receive different natal kicks, leading to different host galaxy offset distributions.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bins_v = np.logspace(np.log10(0.5), np.log10(700), 50)

for lbl, mask, c in bns_classes:
    vals = v_sys_bns[mask]
    ok = np.isfinite(vals) & (vals > 0)
    if ok.sum() == 0:
        continue
    ax1.hist(vals[ok], bins=bins_v, weights=w_bns[mask][ok],
             density=True, histtype='step', lw=2, color=c, label=lbl)

ax1.set_xscale('log')
ax1.set_xlabel(r'$v_\mathrm{sys}$ [km/s]')
ax1.set_ylabel('Weighted PDF')
ax1.set_title('BNS Systemic Velocities', fontweight='bold')
ax1.legend(fontsize=8)

for lbl, mask, c in bhns_classes:
    vals = v_sys_bhns[mask]
    ok = np.isfinite(vals) & (vals > 0)
    if ok.sum() == 0:
        continue
    ax2.hist(vals[ok], bins=bins_v, weights=w_bhns[mask][ok],
             density=True, histtype='step', lw=2, color=c, label=lbl)

ax2.set_xscale('log')
ax2.set_xlabel(r'$v_\mathrm{sys}$ [km/s]')
ax2.set_ylabel('Weighted PDF')
ax2.set_title('BHNS Systemic Velocities', fontweight='bold')
ax2.legend(fontsize=8)

fig.suptitle('Post-SN Systemic Velocity by GRB Class',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/systemic_velocity_distributions.png')
plt.show()


## 11. Ballistic Travel Distance Distributions

Estimated ballistic travel distance from the birth site: $d_\mathrm{travel} \approx v_\mathrm{sys} \times t_\mathrm{delay}$.
This is a **strict upper bound** assuming free (unbound) straight-line motion.
In reality, the host galaxy's gravitational potential decelerates and confines the binary,
reducing actual galactocentric offsets by one to two orders of magnitude
(see Bloom et al. 1999; Fong & Berger 2013 for orbit-integrated predictions).
These distances are useful for **relative** comparisons between GRB classes,
but should not be interpreted as physical host-galaxy offsets.

Cumulative distributions are weighted by STROOPWAFEL weights to produce rate-representative CDFs.


In [ ]:
# Standard astronomy mnemonic: 1 km/s * 1 Myr = 1.0227 pc = 1.0227e-3 kpc
# (Binney & Tremaine 2008 conversion table).  The earlier value 1.0227 kpc
# was off by 1000x and inflated all "ballistic distances" accordingly.
KPC_PER_KM_S_MYR = 1.0227e-3

d_travel_bns  = v_sys_bns  * delay_bns  * KPC_PER_KM_S_MYR  # [kpc]
d_travel_bhns = v_sys_bhns * delay_bhns * KPC_PER_KM_S_MYR

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Weighted CDFs ─────────────────────────────────────────────────────────
for lbl, mask, c in bns_classes:
    vals = d_travel_bns[mask]
    wts  = w_bns[mask]
    ok   = np.isfinite(vals) & (vals > 0) & np.isfinite(wts)
    if ok.sum() < 10:
        continue
    order    = np.argsort(vals[ok])
    sorted_v = vals[ok][order]
    sorted_w = wts[ok][order]
    cdf      = np.cumsum(sorted_w) / np.sum(sorted_w)
    ax1.plot(sorted_v, cdf, color=c, lw=2, label=lbl)

ax1.set_xscale('log')
ax1.set_xlabel('Ballistic travel distance [kpc]  (upper bound)')
ax1.set_ylabel('Cumulative fraction (rate-weighted)')
ax1.set_title('BNS Travel Distance CDF', fontweight='bold')
ax1.legend(fontsize=8)
ax1.set_xlim(1e-3, 1e3)
ax1.axvline(10, color='gray', ls=':', alpha=0.5)
ax1.text(12, 0.5, r'10 kpc (typ. galaxy $R_e$)', color='gray', fontsize=7, rotation=90, va='center')

for lbl, mask, c in bhns_classes:
    vals = d_travel_bhns[mask]
    wts  = w_bhns[mask]
    ok   = np.isfinite(vals) & (vals > 0) & np.isfinite(wts)
    if ok.sum() < 10:
        continue
    order    = np.argsort(vals[ok])
    sorted_v = vals[ok][order]
    sorted_w = wts[ok][order]
    cdf      = np.cumsum(sorted_w) / np.sum(sorted_w)
    ax2.plot(sorted_v, cdf, color=c, lw=2, label=lbl)

ax2.set_xscale('log')
ax2.set_xlabel('Ballistic travel distance [kpc]  (upper bound)')
ax2.set_ylabel('Cumulative fraction (rate-weighted)')
ax2.set_title('BHNS Travel Distance CDF', fontweight='bold')
ax2.legend(fontsize=8)
ax2.set_xlim(1e-3, 1e3)
ax2.axvline(10, color='gray', ls=':', alpha=0.5)

fig.suptitle(r'Ballistic Travel Distance ($v_\mathrm{sys} \times t_\mathrm{delay}$, upper bound)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/ballistic_travel_distance.png')
plt.show()


## 12. Physical Host-Galaxy Offset Distributions

Orbit-integrated projected offsets using a Hernquist (1990) galaxy potential, replacing the ballistic upper bound with a physically motivated model (cf. Bloom et al. 1999, ApJ 518 L1; Fong & Berger 2013, ApJ 776, 18).

**Birth-site sampling.** Each binary's birth radius is drawn from the Hernquist stellar-density profile via inverse-CDF sampling of $M(<r)/M_\mathrm{tot} = (r/(r+a))^2$ (`hernquist_birth_radius`), so birthplaces trace the galaxy light. The draw is bounded at $r_\mathrm{cap} = 1000\,a$ (pileup fraction $\sim 10^{-6}$); the previous $20\,a$ cap clipped 9.3% of draws into a delta function and biased the predicted offset tail.

**Kick direction.** The kick polar angle $\theta_\mathrm{launch}$ is drawn isotropically (Bloom et al. 1999) and threaded through both the analytic energy/apocenter calculation and the numerical orbit integration, so the bound/unbound branching uses the same orbit that is then propagated.

**Host mixture.** Following Fong & Berger (2013): $\sim$75% star-forming ($M_* \sim 10^{9.8}$–$10^{10.5}\,M_\odot$, $R_e \sim 3.6$–$5$ kpc) and $\sim$25% elliptical ($M_* \sim 10^{11}\,M_\odot$, $R_e \sim 8$ kpc). The default single-host call uses $M_\mathrm{gal} = 10^{10.5}\,M_\odot$, $R_e = 5\,$kpc.

**Selection effects.** The Fong & Berger (2013) sample requires both an afterglow detection and an identified host galaxy; "hostless" bursts at large offsets are underrepresented. The model CDF includes all mergers, so it should extend to larger offsets than the observed sample.

Observed short GRB offsets from Fong & Berger (2013, Table 3) are overlaid for comparison. Merger-driven long GRB offsets from Rastinejad et al. (2025) are shown where available.

In [ ]:
from grb_offsets import (compute_offsets_population, compute_offsets_mixed_hosts,
                         compute_offsets_delay_hosts,
                         weighted_offset_cdf, HOST_MODELS,
                         OBSERVED_SGRB_OFFSETS_KPC, OBSERVED_LGRB_MERGER_OFFSETS_KPC)
from scipy.stats import ks_2samp

# Compute physical offsets in mixed Fong & Berger 2013 host populations
bns_classes_off = [
    ('lbGRB (HMNS)',    lbGRB_hmns,   C_LB_HMNS, '-'),
    ('lbGRB (disk)',    lbGRB_disk,   C_LB_DISK, '--'),
    ('Faint lbGRB',    bns_faint_lb, C_FAINT,   ':'),
]

bhns_classes_off = [
    ('lbGRB (BHNS)',       bhns_long,     C_LONG_BH,  '-'),
    ('Faint lbGRB (BHNS)', bhns_faint_lb, C_FAINT_BH, '--'),
]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ── Panel 1: BNS offsets (mixed hosts) ────────────────────────────────────
ax1 = axes[0]
ks_results_bns = {}
for lbl, mask, c, ls in bns_classes_off:
    res = compute_offsets_mixed_hosts(
        v_sys_bns[mask], delay_bns[mask], weights=w_bns[mask], max_systems=30000)
    if len(res['mixed_offsets']) < 10:
        continue
    x, cdf = weighted_offset_cdf(res['mixed_offsets'], res['mixed_weights'])
    ax1.plot(x, cdf, color=c, ls=ls, lw=2, label=lbl)
    # Two-sample KS test against F&B 2013 (unweighted on the model side
    # since scipy.stats.ks_2samp does not support sample weights).
    finite_off = res['mixed_offsets'][np.isfinite(res['mixed_offsets'])]
    if len(finite_off) > 1:
        ks_results_bns[lbl] = ks_2samp(finite_off, OBSERVED_SGRB_OFFSETS_KPC)

obs_sorted = np.sort(OBSERVED_SGRB_OFFSETS_KPC)
obs_cdf = np.arange(1, len(obs_sorted) + 1) / len(obs_sorted)
ax1.step(obs_sorted, obs_cdf, color='#6B21A8', lw=2, ls='-', alpha=0.8,
         where='post', label='Obs. SGRB (F&B 2013)')

# Annotate KS p-value for the dominant class (lbGRB HMNS) vs F&B 2013
if 'lbGRB (HMNS)' in ks_results_bns:
    ks = ks_results_bns['lbGRB (HMNS)']
    ax1.text(0.97, 0.04, f'KS vs F&B13: p = {ks.pvalue:.2g}',
             transform=ax1.transAxes, fontsize=7, ha='right', va='bottom',
             color='#6B21A8',
             bbox=dict(boxstyle='round,pad=0.2', fc='white',
                       ec='#CBD5E1', alpha=0.85))

ax1.set_xlim(0.05, 100)
ax1.set_xscale('log')
ax1.set_xlabel('Projected offset [kpc]')
ax1.set_ylabel('Cumulative fraction')
ax1.set_title('BNS: Mixed-Host Offsets', fontsize=11, pad=8)
ax1.legend(fontsize=7.5, loc='lower right')
ax1.grid(which='both', color='#E2E8F0', lw=0.5, alpha=0.7)
ax1.text(0.03, 0.97, 'Obs. CDF excludes hostless\nbursts at large offsets',
         transform=ax1.transAxes, fontsize=6, color='#6B21A8',
         va='top', style='italic', alpha=0.8)

# ── Panel 2: BHNS offsets (mixed hosts) ───────────────────────────────────
ax2 = axes[1]
for lbl, mask, c, ls in bhns_classes_off:
    res = compute_offsets_mixed_hosts(
        v_sys_bhns[mask], delay_bhns[mask], weights=w_bhns[mask], max_systems=30000)
    if len(res['mixed_offsets']) < 10:
        continue
    x, cdf = weighted_offset_cdf(res['mixed_offsets'], res['mixed_weights'])
    ax2.plot(x, cdf, color=c, ls=ls, lw=2, label=lbl)

if len(OBSERVED_LGRB_MERGER_OFFSETS_KPC) > 0:
    obs_l_sorted = np.sort(OBSERVED_LGRB_MERGER_OFFSETS_KPC)
    obs_l_cdf = np.arange(1, len(obs_l_sorted) + 1) / len(obs_l_sorted)
    ax2.step(obs_l_sorted, obs_l_cdf, color='#6B21A8', lw=2, ls='-', alpha=0.8,
             where='post', label='Obs. lGRB-KN (R+2025)')

ax2.set_xlim(0.05, 100)
ax2.set_xscale('log')
ax2.set_xlabel('Projected offset [kpc]')
ax2.set_ylabel('Cumulative fraction')
ax2.set_title('BHNS: Mixed-Host Offsets', fontsize=11, pad=8)
ax2.legend(fontsize=7.5, loc='lower right')
ax2.grid(which='both', color='#E2E8F0', lw=0.5, alpha=0.7)

# Panel 3: Delay-dependent host assignment (short delays -> SF hosts;
# long delays -> elliptical hosts; Fong & Berger 2013 host demographics)
ax3 = axes[2]
all_mask = np.ones(n_bns, dtype=bool)
res_delay = compute_offsets_delay_hosts(
    v_sys_bns, delay_bns, weights=w_bns, max_systems=25000)
if len(res_delay['offsets_kpc']) > 10:
    x_all, cdf_all = weighted_offset_cdf(
        res_delay['offsets_kpc'], res_delay['weights_sub'])
    ax3.plot(x_all, cdf_all, 'k-', lw=2, label='All BNS (delay-assigned hosts)')

for hname, hc, hls in [('SF_disk', '#059669', '-'),
                        ('SF_massive', '#1D4ED8', '--'),
                        ('Elliptical', '#DC2626', ':')]:
    hmask = res_delay['host_assignments'] == hname
    if hmask.sum() < 10:
        continue
    x_h, cdf_h = weighted_offset_cdf(
        res_delay['offsets_kpc'][hmask], res_delay['weights_sub'][hmask])
    ax3.plot(x_h, cdf_h, color=hc, ls=hls, lw=1.5, label=f'{hname}')

ax3.step(obs_sorted, obs_cdf, color='#6B21A8', lw=2, ls='-', alpha=0.8,
         where='post', label='Obs. SGRB (F&B 2013)')
ax3.set_xlim(0.05, 100)
ax3.set_xscale('log')
ax3.set_xlabel('Projected offset [kpc]')
ax3.set_ylabel('Cumulative fraction')
ax3.set_title('BNS: Delay-Dependent Host Assignment', fontsize=11, pad=8)
ax3.legend(fontsize=7.5, loc='lower right')
ax3.grid(which='both', color='#E2E8F0', lw=0.5, alpha=0.7)

fig.suptitle('Projected Host-Galaxy Offsets (Hernquist Potential, Mixed Hosts)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/projected_offsets.png')
plt.show()

## 13. BH Spin Dependence and Prior Marginalization

BHNS GRB rates are highly sensitive to the assumed BH spin distribution. We compare two priors: a flat prior (uniform in $a$) and a Fuller & Ma (2019) prior (favoring low spins from efficient angular momentum transport). The marginalized rate quantifies the predicted BHNS lbGRB contribution under each assumption.


In [ ]:
iz0 = np.argmin(np.abs(redshifts))
print(f"BHNS lbGRB R(z=0):  Flat prior = {rate_label(r_long_flat[iz0])},"
      f"  Fuller & Ma = {rate_label(r_long_fm19[iz0])} Gpc\u207b\u00b3 yr\u207b\u00b9")
print(f"BHNS faint R(z=0):  Flat prior = {rate_label(r_short_flat[iz0])},"
      f"  Fuller & Ma = {rate_label(r_short_fm19[iz0])} Gpc\u207b\u00b3 yr\u207b\u00b9")
print(f"\nWith misalignment correction (~{MISALIGNMENT_SYSTEMATIC_FACTOR:.0%} reduction):")
print(f"  BHNS lbGRB (F&M, aligned):      {rate_label(r_long_fm19[iz0])}")
print(f"  BHNS lbGRB (F&M, w/ misalign):  "
      f"{rate_label(float(apply_bhns_misalignment(r_long_fm19[iz0])))}")

# BHNS spin x EOS sensitivity using EOS-self-consistent (R_1.4, M_TOV) pairs
# from Read+ 2009 via ns_radius_from_eos, with clip_chi=0.9 to enforce the
# Foucart+ 2018 calibration boundary.
print("\n=== BHNS Spin x EOS Sensitivity (Read+ 2009 EOSs) ===")
for eos in ['APR4', 'SFHo', 'DD2']:
    ep = EOS_MODELS[eos]
    R_NS_eos = ns_radius_from_eos(NS_bh, eos)
    for a_test in [0.0, 0.5, 0.9]:
        cbh_eos = classify_bhns(BH, NS_bh, a_BH=a_test,
                                R_NS_km=R_NS_eos, clip_chi=0.9)
        f_long = 100 * w_bhns[cbh_eos['Long cbGRB']].sum() / w_bhns.sum()
        print(f"  {eos:5s}  a={a_test:.1f}  R_1.4={ep['R_1p4']:.1f} km  "
              f"M_TOV={ep['M_TOV']:.2f}  lbGRB fraction: {f_long:.1f}%")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

for a in spin_grid:
    alpha = 0.3 + 0.15 * a
    ax1.plot(redshifts, spin_rates_long[a], color=C_LONG_BH, alpha=alpha, lw=0.8)

ax1.plot(redshifts, r_long_flat, color=C_LONG_BH, lw=2.5, label='Flat prior (aligned)')
ax1.plot(redshifts, r_long_fm19, color=C_LONG_BH, lw=2.5, ls='--', label='F&M 2019 (aligned)')
ax1.plot(redshifts, apply_bhns_misalignment(r_long_fm19),
         color=C_LONG_BH, lw=2, ls=':', label=f'F&M 2019 (w/ misalign. x{MISALIGNMENT_SYSTEMATIC_FACTOR})')
ax1.plot(redshifts, rate_bhns_all, 'k-', lw=2, alpha=0.5, label='All BHNS')

ax1.set_xlim(0, 10)
ax1.set_yscale('log')
ax1.set_ylim(bottom=0.001)
ax1.set_xlabel('Redshift $z$')
ax1.set_ylabel(r'$\mathcal{R}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax1.set_title('BHNS lbGRB: Spin Marginalization', fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

spin_arr = np.array(spin_grid)
r0_long  = [spin_rates_long[a][iz0] for a in spin_grid]
r0_short = [spin_rates_short[a][iz0] for a in spin_grid]
ax2.plot(spin_arr, r0_long, 'o-', color=C_LONG_BH, lw=2, ms=8,
         label=f'lbGRB ($R_{{1.4}}=12.0$ km)')

eos_ls = {'APR4': ':', 'DD2': '--'}
eos_c  = {'APR4': '#059669', 'DD2': '#7C3AED'}
for eos_name in ['APR4', 'DD2']:
    ep = EOS_MODELS[eos_name]
    R_NS_eos = ns_radius_from_eos(NS_bh, eos_name)
    r0_eos = []
    for a in spin_grid:
        cbh_e = classify_bhns(BH, NS_bh, a_BH=a,
                              R_NS_km=R_NS_eos, clip_chi=0.9)
        r0_eos.append(compute_merger_rate(
            redshifts, times, time_first_SF, n_formed_BHNS,
            dPdlogZ, metallicities, p_draw,
            Z_bhns[cbh_e['Long cbGRB']], delay_bhns[cbh_e['Long cbGRB']],
            w_bhns[cbh_e['Long cbGRB']])[iz0])
    ax2.plot(spin_arr, r0_eos, 's' + eos_ls[eos_name], color=eos_c[eos_name],
             lw=1.5, ms=6, label=f'lbGRB ({eos_name}, $R_{{1.4}}={ep["R_1p4"]}$ km)')

ax2.plot(spin_arr, r0_short, 's--', color=C_FAINT_BH, lw=2, ms=8, label='Faint lbGRB')
ax2.set_xlabel(r'BH spin $a_\mathrm{BH}$')
ax2.set_ylabel(r'$\mathcal{R}(z=0)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax2.set_title('BHNS $z=0$ Rate vs BH Spin (EOS bracket)', fontweight='bold')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.3)

fig.suptitle('BH Spin Parameter Exploration',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/bh_spin_exploration.png')
plt.show()


## 14. Model A vs Model K ($\alpha_\mathrm{CE}$ Sensitivity)

Model K uses $\alpha_\mathrm{CE} = 0.5$ (less efficient common-envelope ejection) compared to Model A ($\alpha_\mathrm{CE} = 1.0$). This shifts the BNS mass and delay-time distributions, altering the GRB class balance.

**CE-efficiency bracket limitation:** We bracket only $\alpha_\mathrm{CE} = 0.5$ (Model K) and $1.0$ (Model A). Broekgaarden et al. (2021, arXiv:2103.02608) show $\sim\mathcal{O}(10)$ rate variation from CE physics. Model G ($\alpha_\mathrm{CE} = 2$) produces some of the highest BHNS rates and would extend the upper bracket. No Model G data is currently available in this pipeline; this limitation should be noted when reporting rate uncertainties.


In [ ]:
bns_K = load_bns(path=DEFAULT_BNS_K_PATH)
m1_K, m2_K = bns_K['m1'], bns_K['m2']
w_K = bns_K['weights']
Z_K = bns_K['metallicity']
delay_K = bns_K['delay_time']

M_tot_K = m1_K + m2_K
q_K = m1_K / m2_K

cls_K_24 = classify_bns_2024(m1_K, m2_K)

n_K = bns_K['n_merging']
print(f"Model K merging BNS: {n_K:,}")
for lbl, m in cls_K_24.items():
    print(f"  {lbl:30s}: {m.sum():>7,}  ({100*m.sum()/n_K:.1f}%)")

expected_rate_K = read_expected_local_rate(DEFAULT_BNS_K_PATH)
mean_mass_K, _ = calibrate_mean_mass_evolved(
    sfr, redshifts, times, time_first_SF,
    dPdlogZ, metallicities, p_draw,
    Z_K, delay_K, w_K, expected_rate_K)
n_formed_K = sfr / mean_mass_K
print(f"Model K: MEAN_MASS = {mean_mass_K:.4e}, expected R(z=0) = {expected_rate_K:.1f}")

merger_rates_K = {}
classes_K = [
    ('lbGRB (HMNS)', cls_K_24['lbGRB + red KN (HMNS)']),
    ('lbGRB (disk)', cls_K_24['lbGRB + red KN (disk)']),
    ('All BNS', np.ones(len(delay_K), dtype=bool)),
]
for label, mask in classes_K:
    rate = compute_merger_rate(
        redshifts, times, time_first_SF, n_formed_K,
        dPdlogZ, metallicities, p_draw,
        Z_K[mask], delay_K[mask], w_K[mask])
    merger_rates_K[label] = rate

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# ── Panel 1: mass distributions ──────────────────────────────────────────
ax = axes[0]
bins_mt = np.linspace(2.0, 5.0, 40)
ax.hist(M_tot, bins=bins_mt, weights=w_bns, density=True,
        histtype='step', lw=2, color='#1D4ED8', label='Model A')
ax.hist(M_tot_K, bins=bins_mt, weights=w_K, density=True,
        histtype='step', lw=2, color='#DC2626', ls='--', label='Model K')
ax.axvline(M_THRESH, color='gray', ls=':', lw=1.5)
ax.set_xlabel(r'$M_\mathrm{tot}$ [$M_\odot$]')
ax.set_ylabel('Weighted PDF')
ax.set_title('$M_\mathrm{tot}$ Distribution', fontweight='bold')
ax.legend(fontsize=9)

# ── Panel 2: class fractions ─────────────────────────────────────────────
ax = axes[1]
cls_A_names = list(cls24.keys())
frac_A = [100 * w_bns[cls24[k]].sum() / w_bns.sum() for k in cls_A_names]
frac_K_vals = [100 * w_K[cls_K_24[k]].sum() / w_K.sum() for k in cls_A_names]

x = np.arange(len(cls_A_names))
width = 0.35
ax.bar(x - width/2, frac_A, width, label='Model A', color='#1D4ED8', alpha=0.8)
ax.bar(x + width/2, frac_K_vals, width, label='Model K', color='#DC2626', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['sbGRB', 'lbGRB\n(HMNS)', 'lbGRB\n(disk)', 'Faint'],
                    fontsize=8)
ax.set_ylabel('Weighted %')
ax.set_title('GRB Class Fractions', fontweight='bold')
ax.legend(fontsize=9)

# ── Panel 3: R(z) comparison ─────────────────────────────────────────────
ax = axes[2]
ax.plot(redshifts, merger_rates_BNS['All BNS'], color='#1D4ED8', lw=2.5, label='Model A: All')
ax.plot(redshifts, merger_rates_K['All BNS'], color='#DC2626', lw=2.5, ls='--', label='Model K: All')
ax.plot(redshifts, merger_rates_BNS['lbGRB (HMNS)'], color='#1D4ED8', lw=1.5, alpha=0.6, label='Model A: lbGRB (HMNS)')
ax.plot(redshifts, merger_rates_K['lbGRB (HMNS)'], color='#DC2626', lw=1.5, ls='--', alpha=0.6, label='Model K: lbGRB (HMNS)')
ax.set_xlim(0, 10)
ax.set_yscale('log')
ax.set_ylim(bottom=0.1)
ax.set_xlabel('Redshift $z$')
ax.set_ylabel(r'$\mathcal{R}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax.set_title('Merger Rate Comparison', fontweight='bold')
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

fig.suptitle('Model A ($\\alpha_\mathrm{CE}=1$) vs Model K ($\\alpha_\mathrm{CE}=0.5$)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('Plots/model_a_vs_k.png')
plt.show()


### Ejecta Mass Predictions by GRB Class

Connecting the Kruger & Foucart (2020, arXiv:2002.07728) dynamical ejecta formulas to the Gottlieb classification. The "blue KN" label implies lanthanide-poor ejecta from the shocked interface (tidal + squeezed), while "red KN" implies lanthanide-rich ejecta from the tidal tail and disk wind. Rastinejad et al. (2025) find the three long-GRB kilonova events have the highest red ejecta masses.

In [ ]:
def weighted_quantile(values, weights, q):
    """Weighted quantile (0-1) via linear interpolation of the weighted CDF."""
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values)
    cw = np.cumsum(weights[order])
    cw /= cw[-1]
    return float(np.interp(q, cw, values[order]))

m_ej_bns = bns_dynamical_ejecta(m1_bns, m2_bns, R_1p4_km=12.0)
m_ej_bhns = bhns_dynamical_ejecta(BH, NS_bh, a_BH=A_BH_FID, R_1p4_km=12.0)

print("=== BNS Dynamical Ejecta by GRB Class ===")
for lbl, mask in [('sbGRB + blue KN', sbGRB_blue),
                  ('lbGRB (HMNS)', lbGRB_hmns),
                  ('lbGRB (disk)', lbGRB_disk),
                  ('Faint / No GRB', faint_or_none),
                  ('All BNS', np.ones(n_bns, dtype=bool))]:
    if mask.sum() == 0:
        print(f"  {lbl:25s}: no systems")
        continue
    ej = m_ej_bns[mask]
    wt = w_bns[mask]
    med = weighted_quantile(ej, wt, 0.5)
    p10 = weighted_quantile(ej, wt, 0.1)
    p90 = weighted_quantile(ej, wt, 0.9)
    print(f"  {lbl:25s}: w.median = {med:.4f} Msun  (10-90%: {p10:.4f}-{p90:.4f})")

print("\n=== BHNS Dynamical Ejecta (a_BH = {:.1f}) ===".format(A_BH_FID))
for lbl, mask in [('lbGRB (massive disk)', bhns_long),
                  ('Faint lbGRB', bhns_faint_lb),
                  ('No GRB', bhns_no_grb)]:
    if mask.sum() == 0:
        print(f"  {lbl:25s}: no systems")
        continue
    ej = m_ej_bhns[mask]
    wt = w_bhns[mask]
    med = weighted_quantile(ej, wt, 0.5)
    print(f"  {lbl:25s}: w.median = {med:.4f} Msun")

print("\nConsistency check:")
_gw170817_ej = float(bns_dynamical_ejecta(1.46, 1.27))
print(f"  GW170817 M_ej_dyn = {_gw170817_ej:.4f} Msun")
assert 0.001 <= _gw170817_ej <= 0.02, (
    f"BNS ejecta sanity check failed: {_gw170817_ej:.4f} Msun "
    "(expected 0.001-0.02 for GW170817-like)")
print(f"  AT2017gfo total ejecta ~ 0.05-0.08 Msun (Rastinejad+ 2025)")
print(f"  (dynamical ejecta is a subset; disk wind adds ~0.01-0.05 Msun)")

## 15. Summary: Key Predictions

Consolidated table of $z=0$ merger rate densities, median properties, and robust trends across the Gottlieb et al. (2024) GRB classification framework applied to COMPAS binary population synthesis.

**Caveats:**
- The zero sbGRB + blue KN rate is an artifact of the COMPAS Model A NS mass floor ($\sim 1.26\,M_\odot$), which places all BNS systems above the HMNS split at $1.2\,M_\mathrm{TOV} \approx 2.46\,M_\odot$. Observed Galactic BNS systems populate this region.
- The "ballistic travel distance" column ($v_\mathrm{sys} \times t_\mathrm{delay}$) is a strict upper bound assuming unbound straight-line motion. Actual host-galaxy offsets would be 1–2 orders of magnitude smaller after accounting for the host gravitational potential (cf. Bloom et al. 1999; Fong & Berger 2013).


In [ ]:
iz0 = np.argmin(np.abs(redshifts))

print("=" * 105)
print(f"{'GRB Class':<30s} {'Type':<12s} {'R(z=0)':>12s} {'Med M_tot':>10s} {'Med t_delay':>12s} {'Med v_sys':>10s} {'Med d_ball':>10s}")
print(f"{'':30s} {'':12s} {'[Gpc-3/yr]':>12s} {'[Msun]':>10s} {'[Myr]':>12s} {'[km/s]':>10s} {'[kpc]*':>10s}")
print("-" * 105)

# Same conversion as cell 26: 1 km/s * 1 Myr = 1.0227 pc = 1.0227e-3 kpc
KPC_PER_KM_S_MYR = 1.0227e-3
kpc_fac = KPC_PER_KM_S_MYR

for lbl, mask_bns_m, rate_key in [
    ('BNS sbGRB + blue KN', sbGRB_blue, 'sbGRB + blue KN'),
    ('BNS lbGRB (HMNS)', lbGRB_hmns, 'lbGRB (HMNS)'),
    ('BNS lbGRB (disk)', lbGRB_disk, 'lbGRB (disk)'),
    ('BNS Faint / No GRB', faint_or_none, 'Faint / No GRB'),
    ('BNS All', np.ones(n_bns, dtype=bool), 'All BNS')]:

    r0 = merger_rates_BNS[rate_key][iz0] if rate_key in merger_rates_BNS else 0
    if mask_bns_m.sum() > 0:
        w_m = w_bns[mask_bns_m]
        mt_med = weighted_quantile(M_tot[mask_bns_m], w_m, 0.5)
        td_med = weighted_quantile(delay_bns[mask_bns_m], w_m, 0.5)
        ok_v = np.isfinite(v_sys_bns[mask_bns_m])
        vs_med = weighted_quantile(v_sys_bns[mask_bns_m][ok_v], w_m[ok_v], 0.5) if ok_v.sum() > 0 else 0
        do_med = vs_med * td_med * kpc_fac
    else:
        mt_med = td_med = vs_med = do_med = 0

    print(f"{lbl:<30s} {'Intrinsic':<12s} {rate_label(r0):>12s} {mt_med:>10.2f} {td_med:>12.0f} {vs_med:>10.1f} {do_med:>10.0f}")

print("-" * 105)

for lbl, mask_bhns_m, r0_val in [
    ('BHNS lbGRB (F&M, aligned)', bhns_long, r_long_fm19[iz0]),
    ('BHNS lbGRB (F&M, misalign)', bhns_long,
        float(apply_bhns_misalignment(r_long_fm19[iz0]))),
    ('BHNS Faint (F&M, aligned)', bhns_faint_lb, r_short_fm19[iz0]),
    ('BHNS All', np.ones(n_bhns, dtype=bool), rate_bhns_all[iz0])]:

    if mask_bhns_m.sum() > 0:
        w_m = w_bhns[mask_bhns_m]
        mt_med = weighted_quantile((BH + NS_bh)[mask_bhns_m], w_m, 0.5)
        td_med = weighted_quantile(delay_bhns[mask_bhns_m], w_m, 0.5)
        ok_v = np.isfinite(v_sys_bhns[mask_bhns_m])
        vs_med = weighted_quantile(v_sys_bhns[mask_bhns_m][ok_v], w_m[ok_v], 0.5) if ok_v.sum() > 0 else 0
        do_med = vs_med * td_med * kpc_fac
    else:
        mt_med = td_med = vs_med = do_med = 0

    print(f"{lbl:<30s} {'Intrinsic':<12s} {rate_label(r0_val):>12s} {mt_med:>10.2f} {td_med:>12.0f} {vs_med:>10.1f} {do_med:>10.0f}")

print("=" * 105)

# Observed (beaming-limited) rates for comparison
print("\n--- Observed (beaming-limited) rates for comparison ---")
for ref, d in OBSERVED_SGRB_RATES.items():
    print(f"  {ref:<30s} {'Observed':<12s} {d['R_obs']:.1f} ({d['R_obs_lo']:.1f}-{d['R_obs_hi']:.1f}) Gpc^-3 yr^-1")
print("  Note: model rates above are INTRINSIC (all-sky). Divide by 1/f_beam to compare.")

print("\n--- Key Findings (upper bounds on GRB fractions) ---")
r_bns_all = merger_rates_BNS['All BNS'][iz0]
r_bns_hmns = merger_rates_BNS['lbGRB (HMNS)'][iz0]
print(f"1. BNS mergers are dominated by lbGRB (HMNS) class: {100*r_bns_hmns/r_bns_all:.0f}% of BNS rate at z=0")
print(f"2. BHNS lbGRB (Fuller & Ma marg.): {rate_label(r_long_fm19[iz0])} Gpc-3/yr "
      f"(with misalignment: ~{rate_label(float(apply_bhns_misalignment(r_long_fm19[iz0])))})")
print(f"3. BNS w.median delay time: {weighted_quantile(delay_bns, w_bns, 0.5):.0f} Myr, "
      f"BHNS: {weighted_quantile(delay_bhns, w_bhns, 0.5):.0f} Myr")
print(f"4. Model K (alpha_CE=0.5) total BNS rate: {rate_label(merger_rates_K['All BNS'][iz0])} vs Model A: {rate_label(r_bns_all)} Gpc-3/yr")


## 16. Beaming Correction: Intrinsic vs Observable GRB Rates

GRB detectors only see jets aimed within the opening angle $\theta_j$ of the line of sight. The observable rate relates to the intrinsic rate by $\mathcal{R}_\mathrm{obs} = f_b \times \mathcal{R}_\mathrm{intrinsic}$, where $f_b = 1 - \cos\theta_j$ is the beaming fraction. For typical sGRB jets ($\theta_j \approx 10°$–$16°$; Fong+ 2015 ApJ 815, 102; Beniamini & Nakar 2019 MNRAS 482, 5430), $f_b \approx 0.015$–$0.04$, meaning only ~2–4% of jets are visible. Below we apply this correction to each GRB class and compare with detector-derived observed rates from Wanderman & Piran (2015), Ghirlanda+ (2016), and Colombo+ (2022).

**Wanderman & Piran (2015) fit branch.** We use the WP15 (MNRAS 448, 3026) Table 2 broken-power-law parameters $R_0 = 4.1$ Gpc$^{-3}$ yr$^{-1}$, $n_1 = 1.7$, $z_\mathrm{peak} = 0.9$, $n_2 = -0.6$ — their preferred fit assuming a Schechter luminosity function for the comoving rate density of *observable* (beamed) sGRBs. WP15 also report $n_2 = -3.4$ for the post-peak slope of the *intrinsic* comoving density under different luminosity-function assumptions; the $-0.6$ branch is the appropriate one when comparing to the model curves below, which already include the beaming factor $f_b$ to produce observable rates.

In [ ]:
# A. Beamed rate table with class-dependent jet half-opening angles
# (sbGRB: Fong+ 2015, Beniamini & Nakar 2019; lbGRB: Gottlieb 2023)
iz0 = np.argmin(np.abs(redshifts))

theta_sb = CLASS_THETA_J['sbGRB']
theta_lb = CLASS_THETA_J['lbGRB']

print(f"{'GRB Class':<28s} {'R_intrinsic':>12s} {'theta_j':>8s} {'R_obs(fid)':>11s} {'R_obs(lo)':>11s} {'R_obs(hi)':>11s}")
print(f"{'':28s} {'[Gpc-3/yr]':>12s} {'[deg]':>8s} {'[Gpc-3/yr]':>11s} {'[Gpc-3/yr]':>11s} {'[Gpc-3/yr]':>11s}")
print("-" * 85)

beaming_rows = [
    ('BNS sbGRB + blue KN',  merger_rates_BNS['sbGRB + blue KN'][iz0], theta_sb),
    ('BNS lbGRB (HMNS)',      merger_rates_BNS['lbGRB (HMNS)'][iz0],   theta_lb),
    ('BNS lbGRB (disk)',      merger_rates_BNS['lbGRB (disk)'][iz0],   theta_lb),
    ('BNS All (sbGRB theta)', merger_rates_BNS['All BNS'][iz0],        theta_sb),
    ('BHNS lbGRB (F&M)',      r_long_fm19[iz0],                        theta_lb),
]

for lbl, r_int, th in beaming_rows:
    r_fid = beamed_rate(r_int, th['fid'])
    r_lo = beamed_rate(r_int, th['lo'])
    r_hi = beamed_rate(r_int, th['hi'])
    print(f"{lbl:<28s} {rate_label(r_int):>12s} {th['fid']:>8.0f} {r_fid:>11.2f} {r_lo:>11.2f} {r_hi:>11.2f}")

print()
print("Observed sGRB rates (beaming-limited, for comparison):")
for ref, d in OBSERVED_SGRB_RATES.items():
    print(f"  {ref}: {d['R_obs']:.1f} ({d['R_obs_lo']:.1f}-{d['R_obs_hi']:.1f}) Gpc^-3 yr^-1")

# B. Main figure: class-dependent beaming vs Wanderman & Piran 2015 R(z)
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

z_mask = redshifts <= 5.0
z_plot = redshifts[z_mask]
wp15_Rz = wanderman_piran_2015_Rz(z_plot)

# ── Panel 1: BNS classes with class-dependent theta_j ────────────────────
ax1 = axes[0]
bns_plot_classes = [
    ('sbGRB + blue KN',  C_SB_BLUE, theta_sb),
    ('lbGRB (HMNS)',      C_LB_HMNS, theta_lb),
    ('lbGRB (disk)',      C_LB_DISK, theta_lb),
]

for lbl, c, th in bns_plot_classes:
    r_z = merger_rates_BNS[lbl]
    r_fid = beamed_rate(r_z, th['fid'])
    ax1.plot(z_plot, r_fid[z_mask], color=c, lw=1.8,
             label=f'{lbl} ($\\theta_j={th["fid"]:.0f}°$)')
    r_lo_env = beamed_rate(r_z, th['lo'])
    r_hi_env = beamed_rate(r_z, th['hi'])
    ax1.fill_between(z_plot, r_lo_env[z_mask], r_hi_env[z_mask],
                     color=c, alpha=0.12)

ax1.plot(z_plot, wp15_Rz['R_best'], color='#6366F1', lw=2.2, ls='--',
         label='WP15 observed $R(z)$', zorder=5)
ax1.fill_between(z_plot, wp15_Rz['R_lo'], wp15_Rz['R_hi'],
                 color='#6366F1', alpha=0.15, zorder=4)

ax1.set_xlim(0, 5)
ax1.set_yscale('log')
ax1.set_ylim(0.001, 300)
ax1.set_xlabel('Redshift $z$')
ax1.set_ylabel(r'$\mathcal{R}_\mathrm{obs}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax1.set_title('BNS: Class-Dependent Beaming', fontweight='bold')
ax1.legend(fontsize=6.5, loc='upper right')
ax1.grid(True, alpha=0.3)

# ── Panel 2: combined BNS + BHNS vs WP15 R(z) ───────────────────────────
ax2 = axes[1]
combined_curves = [
    ('All BNS',              merger_rates_BNS['All BNS'], 'black', theta_sb),
    ('BHNS lbGRB (F&M)',     r_long_fm19,                 C_LONG_BH, theta_lb),
]

for lbl, r_z, c, th in combined_curves:
    ax2.plot(z_plot, r_z[z_mask], color=c, ls=':', lw=1.0, alpha=0.35,
             label=f'{lbl} (intrinsic)')
    r_fid = beamed_rate(r_z, th['fid'])
    ax2.plot(z_plot, r_fid[z_mask], color=c, lw=2.0,
             label=f'{lbl} ($\\theta_j={th["fid"]:.0f}°$)')
    r_lo_env = beamed_rate(r_z, th['lo'])
    r_hi_env = beamed_rate(r_z, th['hi'])
    ax2.fill_between(z_plot, r_lo_env[z_mask], r_hi_env[z_mask],
                     color=c, alpha=0.12)

ax2.plot(z_plot, wp15_Rz['R_best'], color='#6366F1', lw=2.2, ls='--',
         label='WP15 observed $R(z)$', zorder=5)
ax2.fill_between(z_plot, wp15_Rz['R_lo'], wp15_Rz['R_hi'],
                 color='#6366F1', alpha=0.15, zorder=4)

ax2.set_xlim(0, 5)
ax2.set_yscale('log')
ax2.set_ylim(0.01, 5000)
ax2.set_xlabel('Redshift $z$')
ax2.set_ylabel(r'$\mathcal{R}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax2.set_title('Combined: Intrinsic vs Observable', fontweight='bold')
ax2.legend(fontsize=6.5, loc='upper right')
ax2.grid(True, alpha=0.3)

# Panel 3: R(z) shape comparison.  We report the RMS fractional residual
# rather than a chi^2_nu because no per-z error bars on the model exist; if
# desired, an approximate chi^2_red can be formed using the WP15 (R_lo, R_hi)
# envelope as a 1-sigma band.
ax3 = axes[2]
r_bns_all_z = merger_rates_BNS['All BNS']
r_bns_beamed_z = beamed_rate(r_bns_all_z, theta_sb['fid'])

wp15_z0_val = wp15_Rz['R_best'][0]
r_bns_z0_beamed = r_bns_beamed_z[0] if r_bns_beamed_z[0] > 0 else 1.0
norm_factor = wp15_z0_val / r_bns_z0_beamed

ax3.plot(z_plot, r_bns_beamed_z[z_mask] * norm_factor, 'k-', lw=2.5,
         label=f'Model (norm. to WP15 at z=0)')
ax3.plot(z_plot, wp15_Rz['R_best'], color='#6366F1', lw=2.2, ls='--',
         label='WP15 $R(z)$ shape')
ax3.fill_between(z_plot, wp15_Rz['R_lo'], wp15_Rz['R_hi'],
                 color='#6366F1', alpha=0.15)

residual_z = z_plot[(z_plot > 0.1) & (z_plot < 3.0)]
r_model_norm = np.interp(residual_z, z_plot, r_bns_beamed_z[z_mask] * norm_factor)
r_wp15_norm = np.interp(residual_z, z_plot, wp15_Rz['R_best'])
mse_frac = np.sum(((r_model_norm - r_wp15_norm) / r_wp15_norm)**2) / len(residual_z)
rms_frac = np.sqrt(mse_frac)

# Optional: a true chi^2_red using the WP15 envelope as 1-sigma
wp15_lo = np.interp(residual_z, z_plot, wp15_Rz['R_lo'])
wp15_hi = np.interp(residual_z, z_plot, wp15_Rz['R_hi'])
sigma_wp15 = 0.5 * (wp15_hi - wp15_lo)
chi2_red = (np.sum(((r_model_norm - r_wp15_norm) / sigma_wp15)**2)
            / max(1, len(residual_z) - 1))

ax3.set_xlim(0, 3)
ax3.set_yscale('log')
ax3.set_xlabel('Redshift $z$')
ax3.set_ylabel(r'$\mathcal{R}(z)$ [Gpc$^{-3}$ yr$^{-1}$]')
ax3.set_title(f'$R(z)$ Shape Comparison '
              f'(RMS frac. resid. = {rms_frac:.3f}; '
              f'$\\chi^2_\\mathrm{{red}}$ vs WP15 env. = {chi2_red:.2f})',
              fontweight='bold', fontsize=10)
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3)

fig.suptitle('Beaming Correction (Class-Dependent $\\theta_j$) vs Observations',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('Plots/rate_beaming_comparison.png')
plt.show()

# ── C. Summary comparison ────────────────────────────────────────────────
r_bns_all_z0 = merger_rates_BNS['All BNS'][iz0]
r_bns_beamed = beamed_rate(r_bns_all_z0, theta_sb['fid'])
wp15_z0 = wanderman_piran_2015_Rz(np.array([0.0]))
wp15_local = OBSERVED_SGRB_RATES['Wanderman & Piran 2015']
print(f"\n--- Beaming Summary at z=0 ---")
print(f"Predicted intrinsic BNS rate:  {rate_label(r_bns_all_z0)} Gpc^-3 yr^-1")
print(f"Predicted obs. sbGRB (theta_j={theta_sb['fid']}°): {r_bns_beamed:.1f} Gpc^-3 yr^-1  "
      f"[f_beam = {1 - np.cos(np.radians(theta_sb['fid'])):.4f}]")
print(f"WP15 R(z=0):                   {wp15_z0['R_best'][0]:.1f} "
      f"({wp15_z0['R_lo'][0]:.1f}-{wp15_z0['R_hi'][0]:.1f}) Gpc^-3 yr^-1")
ratio = r_bns_beamed / wp15_local['R_obs']
print(f"Predicted / WP15(z=0) ratio:   {ratio:.1f}x")
print(f"R(z) RMS fractional residual (0.1<z<3): {rms_frac:.3f}")
print(f"R(z) chi^2_red vs WP15 envelope (0.1<z<3): {chi2_red:.2f}")